ML2026


Ce notebook est divise en 4 parties :
1. Chargement des donnees et inspection
2. Pretraitement (controle des donnees manquantes / incorrectes)
3. Espace des features (alignement avec la cible + feature engineering polynomial)
4. Entrainement des modeles

## 1) Chargement et Inspection des Donnees

Objectif de cette partie :
- charger les fichiers train/test depuis lenvironnement local
- voir la taille des donnees, les colonnes et les types
- obtenir une premiere vue densemble


In [1]:
import warnings
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVR

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
warnings.filterwarnings('ignore')


def has_xgboost() -> bool:
    try:
        from xgboost import XGBRegressor  # noqa: F401
        return True
    except Exception:
        return False


def require_xgboost_regressor():
    try:
        from xgboost import XGBRegressor
        return XGBRegressor
    except Exception as exc:
        raise ImportError('Impossible dimporter xgboost. xgboost est requis pour cette cellule.') from exc


def has_catboost() -> bool:
    try:
        from catboost import CatBoostRegressor  # noqa: F401
        return True
    except Exception:
        return False


def require_catboost_regressor():
    try:
        from catboost import CatBoostRegressor
        return CatBoostRegressor
    except Exception as exc:
        raise ImportError('Impossible dimporter catboost. catboost est requis pour cette cellule.') from exc


def require_target_encoder_backend():
    try:
        from sklearn.preprocessing import TargetEncoder
        return 'sklearn', TargetEncoder
    except Exception:
        try:
            from category_encoders import TargetEncoder
            return 'category_encoders', TargetEncoder
        except Exception as exc:
            raise ImportError('Impossible dimporter TargetEncoder. `sklearn` ou `category_encoders` est requis.') from exc


In [ ]:
# Configuration du chemin des donnees pour Jupyter local (chemins directs)
TRAIN_PATH = Path('train.csv')
TEST_PATH = Path('test.csv')

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        'Fichiers introuvables. Verifie : ' + str(TRAIN_PATH) + ' ve ' + str(TEST_PATH)
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f'Train path: {TRAIN_PATH}')
print(f'Test path:  {TEST_PATH}')
print(f'Forme train: {train.shape}')
print(f'Forme test:  {test.shape}')


FileNotFoundError: Fichiers introuvables. Verifie : \Users\selimkusursuz\Desktop\ML 2026\ML2026\donnees\train.csv ve \Users\selimkusursuz\Desktop\ML 2026\ML2026\donnees\test.csv

In [ ]:
print('TRAIN - ilk 10 satir')
display(train.head(10))

print('TEST - ilk 10 satir')
display(test.head(10))


TRAIN - ilk 10 satir


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,0,21,female,b.sc,7.91,98.8,no,4.9,average,online videos,low,easy,78.3
1,1,18,other,diploma,4.95,94.8,yes,4.7,poor,self-study,medium,moderate,46.7
2,2,20,female,b.sc,4.68,92.6,yes,5.8,poor,coaching,high,moderate,99.0
3,3,19,male,b.sc,2.00,49.5,yes,8.3,average,group study,high,moderate,63.9
4,4,23,male,bca,7.65,86.9,yes,9.6,good,self-study,high,easy,100.0
5,5,24,male,b.com,5.04,85.1,yes,9.4,average,online videos,medium,moderate,70.1
6,6,20,male,b.sc,4.28,87.0,no,9.1,average,mixed,high,moderate,63.4
7,7,22,female,ba,4.19,44.9,yes,8.8,good,self-study,high,hard,76.8
8,8,22,other,b.com,1.06,98.3,yes,5.0,poor,mixed,low,moderate,46.7
9,9,18,male,bba,3.44,80.9,yes,6.2,good,group study,medium,easy,58.2


TEST - ilk 10 satir


,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate
5,630005,21,male,b.sc,6.22,55.3,yes,4.6,average,mixed,high,moderate
6,630006,22,other,ba,3.59,91.6,yes,7.7,good,mixed,medium,moderate
7,630007,20,male,b.sc,3.14,82.7,no,4.2,average,online videos,high,moderate
8,630008,18,male,diploma,6.49,86.0,yes,7.3,poor,group study,high,easy
9,630009,23,female,b.tech,7.72,95.8,yes,6.4,average,coaching,low,easy


In [ ]:
print('Nombre de colonnes TRAIN :', train.shape[1])
print('Nombre de colonnes TEST :', test.shape[1])

train_cols = set(train.columns)
test_cols = set(test.columns)

only_train = sorted(list(train_cols - test_cols))
only_test = sorted(list(test_cols - train_cols))
common = sorted(list(train_cols & test_cols))

print('\nColonnes uniquement dans TRAIN :', only_train if only_train else 'Yok')
print('Colonnes uniquement dans TEST :', only_test if only_test else 'Yok')
print('Nombre de colonnes communes :', len(common))


Nombre de colonnes TRAIN : 13
Nombre de colonnes TEST : 12

Colonnes uniquement dans TRAIN : ['exam_score']
Colonnes uniquement dans TEST : Yok
Nombre de colonnes communes : 12


In [ ]:
print('--- TRAIN INFO ---')
train.info()

print('\n--- TEST INFO ---')
test.info()


--- TRAIN INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 13 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                630000 non-null  int64  
 1   age               630000 non-null  int64  
 2   gender            630000 non-null  object 
 3   course            630000 non-null  object 
 4   study_hours       630000 non-null  float64
 5   class_attendance  630000 non-null  float64
 6   internet_access   630000 non-null  object 
 7   sleep_hours       630000 non-null  float64
 8   sleep_quality     630000 non-null  object 
 9   study_method      630000 non-null  object 
 10  facility_rating   630000 non-null  object 
 11  exam_difficulty   630000 non-null  object 
 12  exam_score        630000 non-null  float64
dtypes: float64(4), int64(2), object(7)
memory usage: 62.5+ MB

--- TEST INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270000 entries, 0 

In [ ]:
numeric_cols = train.select_dtypes(include=[np.number]).columns.tolist()

print('Colonnes numeriques TRAIN :', numeric_cols)
print('\nResume numerique TRAIN :')
display(train[numeric_cols].describe().T)

if 'exam_score' in train.columns:
    print('\nResume de base de la cible (exam_score) :')
    display(train['exam_score'].describe())


Colonnes numeriques TRAIN : ['id', 'age', 'study_hours', 'class_attendance', 'sleep_hours', 'exam_score']

Resume numerique TRAIN :


,count,mean,std,min,25%,50%,75%,max
id,630000.0,314999.500000,181865.479132,0.000,157499.75,314999.5,472499.25,629999.00
age,630000.0,20.545821,2.260238,17.000,19.00,21.0,23.00,24.00
study_hours,630000.0,4.002337,2.359880,0.080,1.97,4.0,6.05,7.91
class_attendance,630000.0,71.987261,17.430098,40.600,57.00,72.6,87.20,99.40
sleep_hours,630000.0,7.072758,1.744811,4.100,5.60,7.1,8.60,9.90
exam_score,630000.0,62.506672,18.916884,19.599,48.80,62.6,76.30,100.00



Resume de base de la cible (exam_score) :


count    630000.000000
mean         62.506672
std          18.916884
min          19.599000
25%          48.800000
50%          62.600000
75%          76.300000
max         100.000000
Name: exam_score, dtype: float64

In [ ]:
print('--- Veri Analizi Ozet Tablosu ---')

eda_rows = []
for col in train.columns:
    s = train[col]
    eda_rows.append({
        'column': col,
        'dtype': str(s.dtype),
        'n_unique': int(s.nunique(dropna=True)),
        'missing_count': int(s.isna().sum()),
        'missing_ratio': float(s.isna().mean()),
        'sample_values': s.dropna().astype(str).head(3).tolist(),
    })

train_eda_summary_df = pd.DataFrame(eda_rows)
display(train_eda_summary_df)


--- Veri Analizi Ozet Tablosu ---


,column,dtype,n_unique,missing_count,missing_ratio,sample_values
0,id,int64,630000,0,0.0,"[0, 1, 2]"
1,age,int64,8,0,0.0,"[21, 18, 20]"
2,gender,object,3,0,0.0,"[female, other, female]"
3,course,object,7,0,0.0,"[b.sc, diploma, b.sc]"
4,study_hours,float64,792,0,0.0,"[7.91, 4.95, 4.68]"
5,class_attendance,float64,617,0,0.0,"[98.8, 94.8, 92.6]"
6,internet_access,object,2,0,0.0,"[no, yes, yes]"
7,sleep_hours,float64,66,0,0.0,"[4.9, 4.7, 5.8]"
8,sleep_quality,object,3,0,0.0,"[average, poor, poor]"
9,study_method,object,5,0,0.0,"[online videos, self-study, coaching]"


## 2) Pretraitement

Dans cette partie, les donnees **ne sont pas modifiees** ; seuls des controles de qualite sont effectues :
- donnees manquantes
- lignes dupliquees
- coherence des types et des colonnes
- controles de qualite des valeurs numeriques (inf, negatif, mantik disi aralik)
- kategorik tutarlilik kontrolleri


In [ ]:
def missing_summary(df: pd.DataFrame, name: str) -> pd.DataFrame:
    report = pd.DataFrame({
        'column': df.columns,
        'missing_count': df.isna().sum().values
    })
    report['missing_ratio'] = report['missing_count'] / len(df)
    report = report.sort_values(['missing_count', 'column'], ascending=[False, True]).reset_index(drop=True)
    print(f"{name} total de cellules manquantes : {int(report['missing_count'].sum())}")
    return report

train_missing_report = missing_summary(train, 'TRAIN')
test_missing_report = missing_summary(test, 'TEST')

print('\nTRAIN - resume des donnees manquantes (20 premieres)')
display(train_missing_report.head(20))

print('TEST - resume des donnees manquantes (20 premieres)')
display(test_missing_report.head(20))


TRAIN total de cellules manquantes : 0
TEST total de cellules manquantes : 0

TRAIN - resume des donnees manquantes (20 premieres)


,column,missing_count,missing_ratio
0,age,0,0.0
1,class_attendance,0,0.0
2,course,0,0.0
3,exam_difficulty,0,0.0
4,exam_score,0,0.0
5,facility_rating,0,0.0
6,gender,0,0.0
7,id,0,0.0
8,internet_access,0,0.0
9,sleep_hours,0,0.0


TEST - resume des donnees manquantes (20 premieres)


,column,missing_count,missing_ratio
0,age,0,0.0
1,class_attendance,0,0.0
2,course,0,0.0
3,exam_difficulty,0,0.0
4,facility_rating,0,0.0
5,gender,0,0.0
6,id,0,0.0
7,internet_access,0,0.0
8,sleep_hours,0,0.0
9,sleep_quality,0,0.0


In [ ]:
print('--- Duplicate ve Kolon/Tip Kontrolu ---')

train_dup_rows = int(train.duplicated().sum())
test_dup_rows = int(test.duplicated().sum())
print(f'TRAIN duplicate satir sayisi: {train_dup_rows}')
print(f'TEST duplicate satir sayisi : {test_dup_rows}')

if 'id' in train.columns:
    print(f"TRAIN duplicate id sayisi: {int(train['id'].duplicated().sum())}")
if 'id' in test.columns:
    print(f"TEST duplicate id sayisi : {int(test['id'].duplicated().sum())}")

common_cols = sorted(set(train.columns).intersection(set(test.columns)))
dtype_rows = []
for col in common_cols:
    dtype_rows.append({
        'column': col,
        'train_dtype': str(train[col].dtype),
        'test_dtype': str(test[col].dtype),
        'dtype_match': str(train[col].dtype) == str(test[col].dtype),
    })
dtype_check_df = pd.DataFrame(dtype_rows)
print('\nColonnes avec incompatibilite de dtype train/test :')
display(dtype_check_df.loc[~dtype_check_df['dtype_match']])


--- Duplicate ve Kolon/Tip Kontrolu ---
TRAIN duplicate satir sayisi: 0
TEST duplicate satir sayisi : 0
TRAIN duplicate id sayisi: 0
TEST duplicate id sayisi : 0

Colonnes avec incompatibilite de dtype train/test :


,column,train_dtype,test_dtype,dtype_match


In [ ]:
print('--- Sayisal Kalite Kontrolleri ---')

def numeric_quality_checks(df: pd.DataFrame, name: str) -> pd.DataFrame:
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    rows = []
    for col in num_cols:
        s = pd.to_numeric(df[col], errors='coerce')
        rows.append({
            'dataset': name,
            'column': col,
            'inf_count': int(np.isinf(s).sum()),
            'negative_count': int((s < 0).sum()),
            'min': float(s.min()) if s.notna().any() else np.nan,
            'max': float(s.max()) if s.notna().any() else np.nan,
        })
    return pd.DataFrame(rows)

train_num_qc = numeric_quality_checks(train, 'TRAIN')
test_num_qc = numeric_quality_checks(test, 'TEST')
numeric_qc_df = pd.concat([train_num_qc, test_num_qc], ignore_index=True)

print('Colonnes contenant Inf ou des valeurs negatives :')
display(numeric_qc_df.loc[(numeric_qc_df['inf_count'] > 0) | (numeric_qc_df['negative_count'] > 0)])

range_rows = []
if 'exam_score' in train.columns:
    out_of_range = int(((train['exam_score'] < 0) | (train['exam_score'] > 100)).sum())
    range_rows.append({'check': 'exam_score in [0,100] (TRAIN)', 'issue_count': out_of_range})
if 'class_attendance' in train.columns:
    out_of_range = int(((train['class_attendance'] < 0) | (train['class_attendance'] > 100)).sum())
    range_rows.append({'check': 'class_attendance in [0,100] (TRAIN)', 'issue_count': out_of_range})
if 'class_attendance' in test.columns:
    out_of_range = int(((test['class_attendance'] < 0) | (test['class_attendance'] > 100)).sum())
    range_rows.append({'check': 'class_attendance in [0,100] (TEST)', 'issue_count': out_of_range})

range_check_df = pd.DataFrame(range_rows)
print('\nTemel aralik kontrolleri:')
display(range_check_df)


--- Sayisal Kalite Kontrolleri ---
Colonnes contenant Inf ou des valeurs negatives :


,dataset,column,inf_count,negative_count,min,max



Temel aralik kontrolleri:


,check,issue_count
0,"exam_score in [0,100] (TRAIN)",0
1,"class_attendance in [0,100] (TRAIN)",0
2,"class_attendance in [0,100] (TEST)",0


In [ ]:
print('--- Kategorik Tutarlilik Kontrolleri ---')

categorical_cols = train.select_dtypes(include=['object', 'string']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c in test.columns]

cat_rows = []
for col in categorical_cols:
    tr = train[col].astype('string').str.strip().str.lower()
    te = test[col].astype('string').str.strip().str.lower()

    tr_set = set(tr.dropna().unique().tolist())
    te_set = set(te.dropna().unique().tolist())

    unseen_in_train = sorted(list(te_set - tr_set))

    cat_rows.append({
        'column': col,
        'train_unique_count': len(tr_set),
        'test_unique_count': len(te_set),
        'test_unseen_in_train_count': len(unseen_in_train),
        'test_unseen_in_train_sample': unseen_in_train[:5],
        'train_blank_like_count': int((tr == '').sum()),
        'test_blank_like_count': int((te == '').sum()),
    })

categorical_qc_df = pd.DataFrame(cat_rows).sort_values('test_unseen_in_train_count', ascending=False).reset_index(drop=True)
display(categorical_qc_df)

print('La section de pretraitement est terminee : seuls le controle et le rapport ont ete realises, les donnees nont pas ete modifiees.')


--- Kategorik Tutarlilik Kontrolleri ---


,column,train_unique_count,test_unique_count,test_unseen_in_train_count,test_unseen_in_train_sample,train_blank_like_count,test_blank_like_count
0,gender,3,3,0,[],0,0
1,course,7,7,0,[],0,0
2,internet_access,2,2,0,[],0,0
3,sleep_quality,3,3,0,[],0,0
4,study_method,5,5,0,[],0,0
5,facility_rating,3,3,0,[],0,0
6,exam_difficulty,3,3,0,[],0,0


La section de pretraitement est terminee : seuls le controle et le rapport ont ete realises, les donnees nont pas ete modifiees.


## 3) Espace des Features

Dans cette partie :
- la relation des features avec `exam_score` est analysee
- les features faiblement alignees sont listees
- un feature engineering polynomial (degree=2) est applique

Note : dans cette partie, seules lanalyse et la selection de features sont effectuees ; lentrainement des modeles se trouve dans la partie 4.


In [ ]:
target_col = 'exam_score'
id_col = 'id' if 'id' in train.columns else None

if target_col not in train.columns:
    raise ValueError(f'Colonne cible introuvable: {target_col}')

train_feat = train.copy()
test_feat = test.copy()

ordinal_maps = {
    'sleep_quality': {'poor': 1, 'average': 2, 'good': 3},
    'facility_rating': {'low': 1, 'medium': 2, 'high': 3},
    'exam_difficulty': {'easy': 1, 'moderate': 2, 'hard': 3},
    'internet_access': {'no': 0, 'yes': 1},
}

categorical_cols = [
    c for c in train_feat.select_dtypes(include=['object', 'string']).columns
    if c in test_feat.columns and c != target_col
]

encoding_rows = []
for col in categorical_cols:
    tr = train_feat[col].astype('string').str.strip().str.lower()
    te = test_feat[col].astype('string').str.strip().str.lower()

    if col in ordinal_maps:
        mapping = ordinal_maps[col]
        method = 'ordinal_map'
    else:
        categories = sorted(pd.concat([tr, te], ignore_index=True).dropna().unique().tolist())
        mapping = {v: i for i, v in enumerate(categories)}
        method = 'label_map_train_test_union'

    train_feat[col] = tr.map(mapping).fillna(-1).astype('Int64')
    test_feat[col] = te.map(mapping).fillna(-1).astype('Int64')

    encoding_rows.append({
        'column': col,
        'method': method,
        'n_mapped_values': len(mapping),
    })

encoding_report_df = pd.DataFrame(encoding_rows)
print('Encodage termine. Rapport sur la conversion numerique des colonnes categorielles :')
display(encoding_report_df)

feature_cols = [
    c for c in train_feat.columns
    if c in test_feat.columns and c not in {target_col, id_col}
]

# Selon la demande : age, facility_rating, internet_access, gender, course, exam_difficulty sont retires
drop_cols = ['age', 'facility_rating', 'internet_access', 'gender', 'course', 'exam_difficulty']
feature_cols = [c for c in feature_cols if c not in drop_cols]

X_train_all = train_feat[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_test_all = test_feat[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

train_medians = X_train_all.median(numeric_only=True)
X_train_all = X_train_all.fillna(train_medians)
X_test_all = X_test_all.fillna(train_medians)

# k3_f1 korunur: facility_rating * sleep_hours * age
k3_needed_f1 = ['facility_rating', 'sleep_hours', 'age']
if all(col in train_feat.columns for col in k3_needed_f1) and all(col in test_feat.columns for col in k3_needed_f1):
    k3_train_f1 = (
        pd.to_numeric(train_feat['facility_rating'], errors='coerce').astype(float)
        * pd.to_numeric(train_feat['sleep_hours'], errors='coerce').astype(float)
        * pd.to_numeric(train_feat['age'], errors='coerce').astype(float)
    )
    k3_test_f1 = (
        pd.to_numeric(test_feat['facility_rating'], errors='coerce').astype(float)
        * pd.to_numeric(test_feat['sleep_hours'], errors='coerce').astype(float)
        * pd.to_numeric(test_feat['age'], errors='coerce').astype(float)
    )

    X_train_all['k3_f1'] = k3_train_f1.replace([np.inf, -np.inf], np.nan)
    X_test_all['k3_f1'] = k3_test_f1.replace([np.inf, -np.inf], np.nan)

    k3_f1_median = X_train_all['k3_f1'].median(skipna=True)
    if pd.isna(k3_f1_median):
        k3_f1_median = 0.0
    X_train_all['k3_f1'] = X_train_all['k3_f1'].fillna(k3_f1_median)
    X_test_all['k3_f1'] = X_test_all['k3_f1'].fillna(k3_f1_median)

    print('Nouvelle feature ajoutee : k3_f1 = facility_rating * sleep_hours * age')
else:
    print(f'k3_f1 na pas pu etre ajoute, colonnes manquantes : {[c for c in k3_needed_f1 if c not in train_feat.columns]}')

# k3_f2 eklenir: sleep_quality * facility_rating * internet_access
k3_needed_f2 = ['sleep_quality', 'facility_rating', 'internet_access']
if all(col in train_feat.columns for col in k3_needed_f2) and all(col in test_feat.columns for col in k3_needed_f2):
    k3_train_f2 = (
        pd.to_numeric(train_feat['sleep_quality'], errors='coerce').astype(float)
        * pd.to_numeric(train_feat['facility_rating'], errors='coerce').astype(float)
        * pd.to_numeric(train_feat['internet_access'], errors='coerce').astype(float)
    )
    k3_test_f2 = (
        pd.to_numeric(test_feat['sleep_quality'], errors='coerce').astype(float)
        * pd.to_numeric(test_feat['facility_rating'], errors='coerce').astype(float)
        * pd.to_numeric(test_feat['internet_access'], errors='coerce').astype(float)
    )

    X_train_all['k3_f2'] = k3_train_f2.replace([np.inf, -np.inf], np.nan)
    X_test_all['k3_f2'] = k3_test_f2.replace([np.inf, -np.inf], np.nan)

    k3_f2_median = X_train_all['k3_f2'].median(skipna=True)
    if pd.isna(k3_f2_median):
        k3_f2_median = 0.0
    X_train_all['k3_f2'] = X_train_all['k3_f2'].fillna(k3_f2_median)
    X_test_all['k3_f2'] = X_test_all['k3_f2'].fillna(k3_f2_median)

    print('Nouvelle feature ajoutee : k3_f2 = sleep_quality * facility_rating * internet_access')
else:
    print(f'k3_f2 na pas pu etre ajoute, colonnes manquantes : {[c for c in k3_needed_f2 if c not in train_feat.columns]}')

# 1. ek ozellik: study_hours * sleep_hours
if 'study_hours' in X_train_all.columns and 'sleep_hours' in X_train_all.columns and 'study_hours' in X_test_all.columns and 'sleep_hours' in X_test_all.columns:
    X_train_all['study_hours_x_sleep_hours'] = (
        pd.to_numeric(X_train_all['study_hours'], errors='coerce').astype(float)
        * pd.to_numeric(X_train_all['sleep_hours'], errors='coerce').astype(float)
    )
    X_test_all['study_hours_x_sleep_hours'] = (
        pd.to_numeric(X_test_all['study_hours'], errors='coerce').astype(float)
        * pd.to_numeric(X_test_all['sleep_hours'], errors='coerce').astype(float)
    )

    X_train_all['study_hours_x_sleep_hours'] = X_train_all['study_hours_x_sleep_hours'].replace([np.inf, -np.inf], np.nan)
    X_test_all['study_hours_x_sleep_hours'] = X_test_all['study_hours_x_sleep_hours'].replace([np.inf, -np.inf], np.nan)

    sh_sh_median = X_train_all['study_hours_x_sleep_hours'].median(skipna=True)
    if pd.isna(sh_sh_median):
        sh_sh_median = 0.0
    X_train_all['study_hours_x_sleep_hours'] = X_train_all['study_hours_x_sleep_hours'].fillna(sh_sh_median)
    X_test_all['study_hours_x_sleep_hours'] = X_test_all['study_hours_x_sleep_hours'].fillna(sh_sh_median)

    print('Nouvelle feature ajoutee : study_hours_x_sleep_hours = study_hours * sleep_hours')
else:
    print('study_hours_x_sleep_hours na pas pu etre ajoute : study_hours ou sleep_hours est introuvable.')

# 2. ek ozellik: study_hours * sleep_quality
if 'study_hours' in X_train_all.columns and 'sleep_quality' in X_train_all.columns and 'study_hours' in X_test_all.columns and 'sleep_quality' in X_test_all.columns:
    X_train_all['study_hours_x_sleep_quality'] = (
        pd.to_numeric(X_train_all['study_hours'], errors='coerce').astype(float)
        * pd.to_numeric(X_train_all['sleep_quality'], errors='coerce').astype(float)
    )
    X_test_all['study_hours_x_sleep_quality'] = (
        pd.to_numeric(X_test_all['study_hours'], errors='coerce').astype(float)
        * pd.to_numeric(X_test_all['sleep_quality'], errors='coerce').astype(float)
    )

    X_train_all['study_hours_x_sleep_quality'] = X_train_all['study_hours_x_sleep_quality'].replace([np.inf, -np.inf], np.nan)
    X_test_all['study_hours_x_sleep_quality'] = X_test_all['study_hours_x_sleep_quality'].replace([np.inf, -np.inf], np.nan)

    sh_sq_median = X_train_all['study_hours_x_sleep_quality'].median(skipna=True)
    if pd.isna(sh_sq_median):
        sh_sq_median = 0.0
    X_train_all['study_hours_x_sleep_quality'] = X_train_all['study_hours_x_sleep_quality'].fillna(sh_sq_median)
    X_test_all['study_hours_x_sleep_quality'] = X_test_all['study_hours_x_sleep_quality'].fillna(sh_sq_median)

    print('Nouvelle feature ajoutee : study_hours_x_sleep_quality = study_hours * sleep_quality')
else:
    print('study_hours_x_sleep_quality na pas pu etre ajoute : study_hours ou sleep_quality est introuvable.')

y_all = pd.to_numeric(train_feat[target_col], errors='coerce').astype(float)
valid_target = y_all.notna()
X_train_all = X_train_all.loc[valid_target].reset_index(drop=True)
y_all = y_all.loc[valid_target].reset_index(drop=True)

print(f'Matrice de features prete : X_train={X_train_all.shape}, X_test={X_test_all.shape}')


Encodage termine. Rapport sur la conversion numerique des colonnes categorielles :


,column,method,n_mapped_values
0,gender,label_map_train_test_union,3
1,course,label_map_train_test_union,7
2,internet_access,ordinal_map,2
3,sleep_quality,ordinal_map,3
4,study_method,label_map_train_test_union,5
5,facility_rating,ordinal_map,3
6,exam_difficulty,ordinal_map,3


Nouvelle feature ajoutee : k3_f1 = facility_rating * sleep_hours * age
Nouvelle feature ajoutee : k3_f2 = sleep_quality * facility_rating * internet_access
Nouvelle feature ajoutee : study_hours_x_sleep_hours = study_hours * sleep_hours
Nouvelle feature ajoutee : study_hours_x_sleep_quality = study_hours * sleep_quality
Matrice de features prete : X_train=(630000, 9), X_test=(270000, 9)


In [ ]:
def corr_strength_label(abs_corr: float) -> str:
    if abs_corr >= 0.70:
        return 'cok guclu'
    if abs_corr >= 0.50:
        return 'guclu'
    if abs_corr >= 0.30:
        return 'orta'
    if abs_corr >= 0.10:
        return 'zayif'
    return 'non alignee ou tres faible'

corr_rows = []
for col in X_train_all.columns:
    c = float(X_train_all[col].corr(y_all))
    corr_rows.append({
        'feature': col,
        'corr': c,
        'abs_corr': abs(c),
        'direction': 'positive' if c >= 0 else 'negative',
        'strength': corr_strength_label(abs(c)),
    })

feature_corr_df = pd.DataFrame(corr_rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)

# Seuil pratique pour le devoir : si |corr| < 0.10, la feature est non alignee / tres faible
NOT_ALIGNED_THRESHOLD = 0.10
not_aligned_features_df = feature_corr_df[feature_corr_df['abs_corr'] < NOT_ALIGNED_THRESHOLD].reset_index(drop=True)

print('15 features les plus alignees avec le score :')
display(feature_corr_df.head(15))

print(f'Uyumsuz/cok zayif feature sayisi (|corr| < {NOT_ALIGNED_THRESHOLD}): {len(not_aligned_features_df)}')
display(not_aligned_features_df[['feature', 'corr', 'abs_corr', 'strength']])


15 features les plus alignees avec le score :


,feature,corr,abs_corr,direction,strength
0,study_hours,0.762267,0.762267,positive,cok guclu
1,study_hours_x_sleep_hours,0.736172,0.736172,positive,cok guclu
2,study_hours_x_sleep_quality,0.703034,0.703034,positive,cok guclu
3,class_attendance,0.360954,0.360954,positive,orta
4,k3_f2,0.249393,0.249393,positive,zayif
5,sleep_quality,0.236895,0.236895,positive,zayif
6,k3_f1,0.234606,0.234606,positive,zayif
7,study_method,-0.183431,0.183431,negative,zayif
8,sleep_hours,0.167410,0.167410,positive,zayif


Uyumsuz/cok zayif feature sayisi (|corr| < 0.1): 0


,feature,corr,abs_corr,strength


In [ ]:
print('--- Detayli Korelasyon Raporu (Pearson + Spearman) ---')

spearman_rows = []
for col in X_train_all.columns:
    c = float(X_train_all[col].corr(y_all, method='spearman'))
    spearman_rows.append({'feature': col, 'spearman_corr': c, 'spearman_abs_corr': abs(c)})

spearman_corr_df = pd.DataFrame(spearman_rows).sort_values('spearman_abs_corr', ascending=False).reset_index(drop=True)

corr_compare_df = feature_corr_df[['feature', 'corr', 'abs_corr']].merge(
    spearman_corr_df[['feature', 'spearman_corr', 'spearman_abs_corr']],
    on='feature',
    how='left'
).sort_values('abs_corr', ascending=False).reset_index(drop=True)

print('Pearson + Spearman karsilastirma (ilk 20)')
display(corr_compare_df.head(20))

high_corr_df = feature_corr_df[feature_corr_df['abs_corr'] >= 0.50].reset_index(drop=True)
mid_corr_df = feature_corr_df[(feature_corr_df['abs_corr'] >= 0.30) & (feature_corr_df['abs_corr'] < 0.50)].reset_index(drop=True)
low_corr_df = feature_corr_df[feature_corr_df['abs_corr'] < 0.10].reset_index(drop=True)

print('Features fortement alignees (|corr| >= 0.50)')
display(high_corr_df[['feature', 'corr', 'abs_corr', 'strength']])

print('Features moyennement alignees (0.30 <= |corr| < 0.50)')
display(mid_corr_df[['feature', 'corr', 'abs_corr', 'strength']])

print('Features faibles / non alignees (|corr| < 0.10)')
display(low_corr_df[['feature', 'corr', 'abs_corr', 'strength']])

corr_report_dir = Path('notebooks/reports/smart_corr_analysis')
corr_report_dir.mkdir(parents=True, exist_ok=True)
corr_compare_df.to_csv(corr_report_dir / 'correlation_compare_pearson_spearman.csv', index=False)
feature_corr_df[feature_corr_df['abs_corr'] >= 0.10].to_csv(corr_report_dir / 'smart_corr_selected.csv', index=False)
feature_corr_df[feature_corr_df['abs_corr'] < 0.10].to_csv(corr_report_dir / 'smart_corr_dropped.csv', index=False)

print('Rapports de correlation enregistres :', corr_report_dir.resolve())


--- Detayli Korelasyon Raporu (Pearson + Spearman) ---
Pearson + Spearman karsilastirma (ilk 20)


,feature,corr,abs_corr,spearman_corr,spearman_abs_corr
0,study_hours,0.762267,0.762267,0.769757,0.769757
1,study_hours_x_sleep_hours,0.736172,0.736172,0.757375,0.757375
2,study_hours_x_sleep_quality,0.703034,0.703034,0.740075,0.740075
3,class_attendance,0.360954,0.360954,0.351515,0.351515
4,k3_f2,0.249393,0.249393,0.241527,0.241527
5,sleep_quality,0.236895,0.236895,0.229405,0.229405
6,k3_f1,0.234606,0.234606,0.229810,0.229810
7,study_method,-0.183431,0.183431,-0.176620,0.176620
8,sleep_hours,0.167410,0.167410,0.160048,0.160048


Features fortement alignees (|corr| >= 0.50)


,feature,corr,abs_corr,strength
0,study_hours,0.762267,0.762267,cok guclu
1,study_hours_x_sleep_hours,0.736172,0.736172,cok guclu
2,study_hours_x_sleep_quality,0.703034,0.703034,cok guclu


Features moyennement alignees (0.30 <= |corr| < 0.50)


,feature,corr,abs_corr,strength
0,class_attendance,0.360954,0.360954,orta


Features faibles / non alignees (|corr| < 0.10)


,feature,corr,abs_corr,strength


Rapports de correlation enregistres : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/smart_corr_analysis


In [ ]:
# Polynomial hesaplamayi sade ve hizli tutmak icin
TOP_BASE_N = 11
POLY_DEGREE = 3
POLY_SAMPLE_N = min(200000, len(X_train_all))
TOP_ENGINEERED_KEEP = 60

base_candidates = feature_corr_df.loc[feature_corr_df['abs_corr'] >= 0.10, 'feature'].head(TOP_BASE_N).tolist()
if len(base_candidates) < 2:
    base_candidates = feature_corr_df['feature'].head(max(2, min(TOP_BASE_N, len(feature_corr_df)))).tolist()

if len(base_candidates) < 2:
    raise ValueError('Pas assez de features pour le modele polynomial.')

if len(X_train_all) > POLY_SAMPLE_N:
    sample_idx = X_train_all.sample(n=POLY_SAMPLE_N, random_state=42).index
    X_poly_base = X_train_all.loc[sample_idx, base_candidates].reset_index(drop=True)
    y_poly = y_all.loc[sample_idx].reset_index(drop=True)
else:
    X_poly_base = X_train_all[base_candidates].reset_index(drop=True)
    y_poly = y_all.reset_index(drop=True)

poly = PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)
X_poly = poly.fit_transform(X_poly_base)
poly_names = poly.get_feature_names_out(base_candidates)

poly_rows = []
y_np = y_poly.to_numpy(dtype=float)
for i, name in enumerate(poly_names):
    vals = X_poly[:, i]
    c = np.corrcoef(vals, y_np)[0, 1]
    if np.isfinite(c):
        poly_rows.append({
            'feature': name,
            'corr': float(c),
            'abs_corr': float(abs(c)),
            'feature_type': 'base' if name in base_candidates else 'engineered',
        })

poly_corr_df = pd.DataFrame(poly_rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)
best_engineered_poly_df = poly_corr_df[poly_corr_df['feature_type'] == 'engineered'].head(TOP_ENGINEERED_KEEP).reset_index(drop=True)

print(f'Polynomial analiz sample boyutu: {len(X_poly_base)}')
print(f'Features de base selectionnees : {base_candidates}')
print('\nPrincipales features polynomiales construites :')
display(best_engineered_poly_df)

selected_base_features = base_candidates
selected_poly_features = best_engineered_poly_df['feature'].tolist()

print('\nListes pretes pour la modelisation dans la partie 4 :')
print('selected_base_features')
print('selected_poly_features')

Polynomial analiz sample boyutu: 200000
Features de base selectionnees : ['study_hours', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'class_attendance', 'k3_f2', 'sleep_quality', 'k3_f1', 'study_method', 'sleep_hours']

Principales features polynomiales construites :


,feature,corr,abs_corr,feature_type
0,study_hours class_attendance,0.796456,0.796456,engineered
1,study_hours class_attendance sleep_hours,0.769122,0.769122,engineered
2,study_hours_x_sleep_hours class_attendance,0.769122,0.769122,engineered
3,study_hours^2 class_attendance,0.762895,0.762895,engineered
4,study_hours study_hours_x_sleep_hours class_at...,0.746170,0.746170,engineered
5,study_hours_x_sleep_quality class_attendance,0.741818,0.741818,engineered
6,study_hours class_attendance sleep_quality,0.741818,0.741818,engineered
7,study_hours class_attendance^2,0.740302,0.740302,engineered
8,study_hours^2,0.735753,0.735753,engineered
9,study_hours sleep_hours,0.735381,0.735381,engineered



Listes pretes pour la modelisation dans la partie 4 :
selected_base_features
selected_poly_features


In [ ]:
# 3.x Analyse des Features (k=3) - generation de nouvelles features a partir du groupe faible / non aligne

if 'X_train_all' not in globals() or 'X_test_all' not in globals() or 'y_all' not in globals():
    raise ValueError('Les cellules precedentes de la partie 3 doivent etre executees (`X_train_all`, `X_test_all`, `y_all` requis).')

seed_features = [
    'sleep_quality',
    'facility_rating',
    'study_method',
    'sleep_hours',
    'age',
    'exam_difficulty',
    'gender',
    'course',
    'internet_access',
]

available_seed_features = [c for c in seed_features if c in X_train_all.columns and c in X_test_all.columns]
missing_seed_features = [c for c in seed_features if c not in available_seed_features]

k = 3
if len(available_seed_features) < k:
    raise ValueError(f'k={k} analyse exige au moins {k} features. Trouvees : {available_seed_features}')

y_np = y_all.to_numpy(dtype=float)

def safe_corr(x: np.ndarray, y: np.ndarray) -> float:
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 3:
        return np.nan
    xv = x[mask].astype(float, copy=False)
    yv = y[mask].astype(float, copy=False)
    if float(np.std(xv)) <= 1e-12 or float(np.std(yv)) <= 1e-12:
        return np.nan
    return float(np.corrcoef(xv, yv)[0, 1])

rows = []
for combo in combinations(available_seed_features, k):
    combo_name = ' * '.join(combo)
    values = X_train_all[list(combo)].prod(axis=1).to_numpy(dtype=float)
    corr = safe_corr(values, y_np)
    rows.append({
        'k': k,
        'feature': combo_name,
        'corr': corr,
        'abs_corr': abs(corr) if pd.notna(corr) else np.nan,
    })

k3_feature_analysis_df = pd.DataFrame(rows).sort_values('abs_corr', ascending=False).reset_index(drop=True)

print('Features seed utilisees :', available_seed_features)
if missing_seed_features:
    print('Features seed introuvables :', missing_seed_features)
print(f'k={k} kombinasyon sayisi:', len(k3_feature_analysis_df))
display(k3_feature_analysis_df.head(20))

# Nouvelles features a partir des 3 combinaisons les plus significatives
TOP_K3_NEW_FEATURES = 3
X_train_k3 = X_train_all.copy()
X_test_k3 = X_test_all.copy()
created_k3_features = []

for rank, (_, row) in enumerate(k3_feature_analysis_df.head(TOP_K3_NEW_FEATURES).iterrows(), start=1):
    if pd.isna(row['corr']):
        continue
    combo_cols = row['feature'].split(' * ')
    feature_name = f'k3_meaningful_feature_{rank}'
    X_train_k3[feature_name] = X_train_all[combo_cols].prod(axis=1).astype(float)
    X_test_k3[feature_name] = X_test_all[combo_cols].prod(axis=1).astype(float)
    created_k3_features.append({
        'feature_name': feature_name,
        'combo': row['feature'],
        'corr': float(row['corr']),
        'abs_corr': float(row['abs_corr']),
    })

created_k3_features_df = pd.DataFrame(created_k3_features)

if not created_k3_features_df.empty:
    print('Nouvelles features creees :')
    display(created_k3_features_df)
else:
    print('Aucune correlation valide trouvee ; aucune nouvelle feature na ete creee.')

Features seed utilisees : ['sleep_quality', 'study_method', 'sleep_hours']
Features seed introuvables : ['facility_rating', 'age', 'exam_difficulty', 'gender', 'course', 'internet_access']
k=3 kombinasyon sayisi: 1


,k,feature,corr,abs_corr
0,3,sleep_quality * study_method * sleep_hours,0.001531,0.001531


Nouvelles features creees :


,feature_name,combo,corr,abs_corr
0,k3_meaningful_feature_1,sleep_quality * study_method * sleep_hours,0.001531,0.001531


In [ ]:
# 3.y Feature Selection (8 / 11 / 15)
# Uretilen feature listesinden (engineered) secim

if 'best_engineered_poly_df' not in globals() or best_engineered_poly_df.empty:
    raise ValueError('La liste de features generees est introuvable. Execute dabord la cellule polynomial.')

candidate_features = best_engineered_poly_df['feature'].dropna().astype(str).tolist()
candidate_features = list(dict.fromkeys(candidate_features))

selected_features_8 = candidate_features[:8]
selected_features_11 = candidate_features[:11]
selected_features_15 = candidate_features[:15]

selection_report_df = pd.DataFrame({
    'set_name': ['top_8', 'top_11', 'top_15'],
    'n_features': [len(selected_features_8), len(selected_features_11), len(selected_features_15)],
})

print('Resume de la selection de features :')
display(selection_report_df)

print('Top 8 feature set:')
print(selected_features_8)

print('Top 11 feature set:')
print(selected_features_11)

print('Top 15 feature set:')
print(selected_features_15)

feature_selection_dir = Path('notebooks/reports/feature_selection_sets')
feature_selection_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame({'feature': selected_features_8}).to_csv(feature_selection_dir / 'selected_features_top8.csv', index=False)
pd.DataFrame({'feature': selected_features_11}).to_csv(feature_selection_dir / 'selected_features_top11.csv', index=False)
pd.DataFrame({'feature': selected_features_15}).to_csv(feature_selection_dir / 'selected_features_top15.csv', index=False)

print('Enregistre :', feature_selection_dir.resolve())


Resume de la selection de features :


,set_name,n_features
0,top_8,8
1,top_11,11
2,top_15,15


Top 8 feature set:
['study_hours class_attendance', 'study_hours class_attendance sleep_hours', 'study_hours_x_sleep_hours class_attendance', 'study_hours^2 class_attendance', 'study_hours study_hours_x_sleep_hours class_attendance', 'study_hours_x_sleep_quality class_attendance', 'study_hours class_attendance sleep_quality', 'study_hours class_attendance^2']
Top 11 feature set:
['study_hours class_attendance', 'study_hours class_attendance sleep_hours', 'study_hours_x_sleep_hours class_attendance', 'study_hours^2 class_attendance', 'study_hours study_hours_x_sleep_hours class_attendance', 'study_hours_x_sleep_quality class_attendance', 'study_hours class_attendance sleep_quality', 'study_hours class_attendance^2', 'study_hours^2', 'study_hours sleep_hours', 'study_hours study_hours_x_sleep_quality class_attendance']
Top 15 feature set:
['study_hours class_attendance', 'study_hours class_attendance sleep_hours', 'study_hours_x_sleep_hours class_attendance', 'study_hours^2 class_attenda

## 4) Entrainement des Modeles

Dans cette partie, nous testons 10 modeles separement sur les donnees de base.
Chaque modele a sa propre cellule.

Flux :
1. Execute dabord la cellule de preparation
2. Execute ensuite les cellules des modeles une par une
3. A la fin, execute la cellule du tableau des resultats et celle de sauvegarde


In [ ]:
HAS_XGB = has_xgboost()
HAS_CAT = has_catboost()
XGBRegressor = require_xgboost_regressor() if HAS_XGB else None
CatBoostRegressor = require_catboost_regressor() if HAS_CAT else None
if 'X_train_all' not in globals() or 'y_all' not in globals():
    raise ValueError('La preparation des features de la partie 3 est introuvable. Execute dabord la partie 3.')

SEED = 42
N_SPLITS = 3

X_basic = X_train_all.copy()
y_basic = y_all.copy()

BASIC_SAMPLE_N = min(120000, len(X_basic))
if len(X_basic) > BASIC_SAMPLE_N:
    sample_idx = X_basic.sample(n=BASIC_SAMPLE_N, random_state=SEED).index
    X_bench = X_basic.loc[sample_idx].reset_index(drop=True)
    y_bench = y_basic.loc[sample_idx].reset_index(drop=True)
else:
    X_bench = X_basic.reset_index(drop=True)
    y_bench = y_basic.reset_index(drop=True)

def evaluate_with_cv(model, X, y, n_splits=3, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rmses, maes = [], []
    for tr_idx, va_idx in kf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model.fit(X_tr, y_tr)
        pred = np.clip(np.asarray(model.predict(X_va), dtype=float), 0.0, 100.0)
        rmses.append(float(np.sqrt(mean_squared_error(y_va, pred))))
        maes.append(float(mean_absolute_error(y_va, pred)))
    return float(np.mean(rmses)), float(np.mean(maes))

model_results = {}

def save_model_result(model_name, status, rmse=np.nan, mae=np.nan, note=''):
    model_results[model_name] = {
        'model': model_name,
        'cv_rmse': rmse,
        'cv_mae': mae,
        'status': status,
        'note': note,
    }

print(f'Taille des donnees de benchmark : X={X_bench.shape}, y={y_bench.shape}')
print(f'XGBoost mevcut mu? {HAS_XGB}')
print(f'CatBoost mevcut mu? {HAS_CAT}')

Taille des donnees de benchmark : X=(120000, 9), y=(120000,)
XGBoost mevcut mu? True
CatBoost mevcut mu? True


In [ ]:
# 1) DummyRegressor
model_name = 'DummyRegressor_mean'
model = DummyRegressor(strategy='mean')
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


DummyRegressor_mean -> RMSE: 18.841655324089864 | MAE: 15.477158573782498


In [ ]:
# 2) LinearRegression
model_name = 'LinearRegression'
model = LinearRegression()
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


LinearRegression -> RMSE: 9.123148527420332 | MAE: 7.291728216048554


In [ ]:
# 3) Ridge
model_name = 'Ridge'
model = Ridge(alpha=1.0, random_state=SEED)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


Ridge -> RMSE: 9.123147783793245 | MAE: 7.2917273832287846


In [ ]:
# 4) Lasso
model_name = 'Lasso'
model = Lasso(alpha=0.001, random_state=SEED, max_iter=2000)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


Lasso -> RMSE: 9.123135682366403 | MAE: 7.291716164920804


In [ ]:
# 5) ElasticNet
model_name = 'ElasticNet'
model = ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=SEED, max_iter=2000)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


ElasticNet -> RMSE: 9.123119772776846 | MAE: 7.2916954218875025


In [ ]:
# 6) RandomForest
model_name = 'RandomForest'
model = RandomForestRegressor(n_estimators=120, max_depth=None, random_state=SEED, n_jobs=-1)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


RandomForest -> RMSE: 9.197099544824427 | MAE: 7.325972834510581


In [ ]:
# 7) ExtraTrees
model_name = 'ExtraTrees'
model = ExtraTreesRegressor(n_estimators=120, max_depth=None, random_state=SEED, n_jobs=-1)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


ExtraTrees -> RMSE: 9.356079548359784 | MAE: 7.448276938125001


In [ ]:
# 8) HistGradientBoosting
model_name = 'HistGradientBoosting'
model = HistGradientBoostingRegressor(learning_rate=0.05, max_depth=None, max_iter=200, random_state=SEED)
rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


HistGradientBoosting -> RMSE: 8.857359451010408 | MAE: 7.072246743280594


In [ ]:
# 9) XGBoost
model_name = 'XGBoost'
if not HAS_XGB:
    save_model_result(model_name, status='skip (paket yok)', note='xgboost import edilemedi')
    print(model_name, '-> skip (paket yok)')
else:
    model = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, objective='reg:squarederror',
        tree_method='hist', random_state=SEED, n_jobs=-1
    )
    rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
    save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
    print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


XGBoost -> RMSE: 8.834721732130822 | MAE: 7.0488476102550495


In [ ]:
# 10) CatBoost
model_name = 'CatBoost'
if not HAS_CAT:
    save_model_result(model_name, status='skip (paket yok)', note='catboost import edilemedi')
    print(model_name, '-> skip (paket yok)')
else:
    model = CatBoostRegressor(
        iterations=600, learning_rate=0.05, depth=6, random_seed=SEED,
        verbose=False, allow_writing_files=False
    )
    rmse, mae = evaluate_with_cv(model, X_bench, y_bench, n_splits=N_SPLITS, seed=SEED)
    save_model_result(model_name, status='ok', rmse=rmse, mae=mae)
    print(model_name, '-> RMSE:', rmse, '| MAE:', mae)


CatBoost -> RMSE: 8.832847892071461 | MAE: 7.050308465980209


In [ ]:
if not model_results:
    raise ValueError('Aucun resultat de modele pour le moment. Execute les cellules des modeles.')

benchmark_results_df = pd.DataFrame(model_results.values())
benchmark_results_df = benchmark_results_df.sort_values(['status', 'cv_rmse'], ascending=[True, True]).reset_index(drop=True)

print('Temel 10-model benchmark sonucu:')
display(benchmark_results_df)


Temel 10-model benchmark sonucu:


,model,cv_rmse,cv_mae,status,note
0,CatBoost,8.832848,7.050308,ok,
1,XGBoost,8.834722,7.048848,ok,
2,HistGradientBoosting,8.857359,7.072247,ok,
3,ElasticNet,9.123120,7.291695,ok,
4,Lasso,9.123136,7.291716,ok,
5,Ridge,9.123148,7.291727,ok,
6,LinearRegression,9.123149,7.291728,ok,
7,RandomForest,9.197100,7.325973,ok,
8,ExtraTrees,9.356080,7.448277,ok,
9,DummyRegressor_mean,18.841655,15.477159,ok,


In [ ]:
if 'benchmark_results_df' not in globals():
    raise ValueError('Execute dabord la cellule du tableau des resultats.')

out_dir = Path('notebooks/reports/basic_10_model_benchmark')
out_dir.mkdir(parents=True, exist_ok=True)

benchmark_results_df.to_csv(out_dir / 'benchmark_results.csv', index=False)
run_info = pd.DataFrame([
    {
        'rows_used': len(X_bench),
        'feature_count': X_bench.shape[1],
        'n_splits': N_SPLITS,
        'target_col': target_col,
        'notes': 'Donnees de base + reglages de base des modeles ; chaque modele a ete execute dans une cellule separee.'
    }
])
run_info.to_csv(out_dir / 'run_info.csv', index=False)

print('Enregistre :', out_dir.resolve())

Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/basic_10_model_benchmark


## 5) Travaux Precedents (Legacy)

Les deux travaux precedents sont reunis ici :
1. Essai de modele polynomial
2. Essai de modele XGBoost

Note : cette zone est independante du benchmark de base de la partie 4.


In [ ]:
# 5.0 Fonctions Partagees
LEGACY_DROP_COLS = ['age', 'facility_rating', 'internet_access', 'gender', 'course', 'exam_difficulty']
LEGACY_ORDINAL_MAPS = {
    'sleep_quality': {'poor': 1, 'average': 2, 'good': 3},
    'facility_rating': {'low': 1, 'medium': 2, 'high': 3},
    'exam_difficulty': {'easy': 1, 'moderate': 2, 'hard': 3},
    'internet_access': {'no': 0, 'yes': 1},
}

_LEGACY_CONTEXT_CACHE = None
_LEGACY_FEATURE_BUNDLE_CACHE = None
_SIMPLE_TRIM_BUNDLE_CACHE = None
_COURSE_STYLE_CONTEXT_CACHE = None


def find_local_data_dir():
    cwd = Path.cwd().resolve()
    search_roots = [cwd, cwd.parent]
    candidate_dirs = []
    for root in search_roots:
        candidate_dirs.extend([root / 'donnees', root / 'donnes', root])

    unique_candidate_dirs = []
    seen = set()
    for d in candidate_dirs:
        key = str(d.resolve())
        if key not in seen:
            seen.add(key)
            unique_candidate_dirs.append(d)

    for d in unique_candidate_dirs:
        train_path = d / 'train.csv'
        test_path = d / 'test.csv'
        if train_path.exists() and test_path.exists():
            return d, train_path, test_path

    searched = ', '.join(str(d.resolve()) for d in unique_candidate_dirs)
    raise FileNotFoundError('train.csv et test.csv introuvables. Emplacements recherches : ' + searched)


def load_local_train_test(verbose=False):
    global_vars = globals()
    if 'train' in global_vars and 'test' in global_vars:
        train_df = global_vars['train'].copy()
        test_df = global_vars['test'].copy()
        data_dir = global_vars.get('DATA_DIR')
        train_path = global_vars.get('TRAIN_PATH')
        test_path = global_vars.get('TEST_PATH')

        if data_dir is None or train_path is None or test_path is None:
            data_dir, train_path, test_path = find_local_data_dir()

        if verbose:
            print(f'DATA_DIR   : {Path(data_dir).resolve()}')
            print(f'Train path : {Path(train_path).resolve()}')
            print(f'Test path  : {Path(test_path).resolve()}')
            print(f'Forme train: {train_df.shape}')
            print(f'Forme test : {test_df.shape}')
        return train_df, test_df, Path(data_dir), Path(train_path), Path(test_path)

    data_dir, train_path, test_path = find_local_data_dir()
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    if verbose:
        print(f'DATA_DIR   : {data_dir.resolve()}')
        print(f'Train path : {train_path.resolve()}')
        print(f'Test path  : {test_path.resolve()}')
        print(f'Forme train: {train_df.shape}')
        print(f'Forme test : {test_df.shape}')
    return train_df, test_df, data_dir, train_path, test_path


def _encode_train_test_union(train_raw, test_raw, target_col):
    train_feat = train_raw.copy()
    test_feat = test_raw.copy()

    categorical_cols = [
        c for c in train_feat.select_dtypes(include=['object', 'string']).columns
        if c in test_feat.columns and c != target_col
    ]

    encoding_rows = []
    for col in categorical_cols:
        tr = train_feat[col].astype('string').str.strip().str.lower()
        te = test_feat[col].astype('string').str.strip().str.lower()

        if col in LEGACY_ORDINAL_MAPS:
            mapping = LEGACY_ORDINAL_MAPS[col]
            method = 'ordinal_map'
        else:
            categories = sorted(pd.concat([tr, te], ignore_index=True).dropna().unique().tolist())
            mapping = {v: i for i, v in enumerate(categories)}
            method = 'label_map_train_test_union'

        train_feat[col] = tr.map(mapping).fillna(-1).astype('Int64')
        test_feat[col] = te.map(mapping).fillna(-1).astype('Int64')
        encoding_rows.append({'column': col, 'method': method, 'n_mapped_values': len(mapping)})

    return train_feat, test_feat, pd.DataFrame(encoding_rows)


def get_legacy_context(refresh=False, verbose=False):
    global _LEGACY_CONTEXT_CACHE
    if _LEGACY_CONTEXT_CACHE is not None and not refresh:
        return _LEGACY_CONTEXT_CACHE

    train_raw, test_raw, data_dir, train_path, test_path = load_local_train_test(verbose=verbose)
    target_col = 'exam_score'
    id_col = 'id' if 'id' in train_raw.columns else None

    if target_col not in train_raw.columns:
        raise ValueError(f'Colonne cible introuvable: {target_col}')

    train_feat, test_feat, encoding_report_df = _encode_train_test_union(train_raw, test_raw, target_col)

    feature_cols = [
        c for c in train_feat.columns
        if c in test_feat.columns and c not in {target_col, id_col}
    ]
    feature_cols = [c for c in feature_cols if c not in LEGACY_DROP_COLS]

    X_train_all = train_feat[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    X_test_all = test_feat[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

    train_medians = X_train_all.median(numeric_only=True)
    X_train_all = X_train_all.fillna(train_medians)
    X_test_all = X_test_all.fillna(train_medians)

    k3_needed_f1 = ['facility_rating', 'sleep_hours', 'age']
    if all(col in train_feat.columns for col in k3_needed_f1) and all(col in test_feat.columns for col in k3_needed_f1):
        X_train_all['k3_f1'] = (
            pd.to_numeric(train_feat['facility_rating'], errors='coerce').astype(float)
            * pd.to_numeric(train_feat['sleep_hours'], errors='coerce').astype(float)
            * pd.to_numeric(train_feat['age'], errors='coerce').astype(float)
        )
        X_test_all['k3_f1'] = (
            pd.to_numeric(test_feat['facility_rating'], errors='coerce').astype(float)
            * pd.to_numeric(test_feat['sleep_hours'], errors='coerce').astype(float)
            * pd.to_numeric(test_feat['age'], errors='coerce').astype(float)
        )
        med = X_train_all['k3_f1'].replace([np.inf, -np.inf], np.nan).median(skipna=True)
        if pd.isna(med):
            med = 0.0
        X_train_all['k3_f1'] = X_train_all['k3_f1'].replace([np.inf, -np.inf], np.nan).fillna(med)
        X_test_all['k3_f1'] = X_test_all['k3_f1'].replace([np.inf, -np.inf], np.nan).fillna(med)

    k3_needed_f2 = ['sleep_quality', 'facility_rating', 'internet_access']
    if all(col in train_feat.columns for col in k3_needed_f2) and all(col in test_feat.columns for col in k3_needed_f2):
        X_train_all['k3_f2'] = (
            pd.to_numeric(train_feat['sleep_quality'], errors='coerce').astype(float)
            * pd.to_numeric(train_feat['facility_rating'], errors='coerce').astype(float)
            * pd.to_numeric(train_feat['internet_access'], errors='coerce').astype(float)
        )
        X_test_all['k3_f2'] = (
            pd.to_numeric(test_feat['sleep_quality'], errors='coerce').astype(float)
            * pd.to_numeric(test_feat['facility_rating'], errors='coerce').astype(float)
            * pd.to_numeric(test_feat['internet_access'], errors='coerce').astype(float)
        )
        med = X_train_all['k3_f2'].replace([np.inf, -np.inf], np.nan).median(skipna=True)
        if pd.isna(med):
            med = 0.0
        X_train_all['k3_f2'] = X_train_all['k3_f2'].replace([np.inf, -np.inf], np.nan).fillna(med)
        X_test_all['k3_f2'] = X_test_all['k3_f2'].replace([np.inf, -np.inf], np.nan).fillna(med)

    if all(col in X_train_all.columns for col in ['study_hours', 'sleep_hours']) and all(col in X_test_all.columns for col in ['study_hours', 'sleep_hours']):
        X_train_all['study_hours_x_sleep_hours'] = (
            pd.to_numeric(X_train_all['study_hours'], errors='coerce').astype(float)
            * pd.to_numeric(X_train_all['sleep_hours'], errors='coerce').astype(float)
        )
        X_test_all['study_hours_x_sleep_hours'] = (
            pd.to_numeric(X_test_all['study_hours'], errors='coerce').astype(float)
            * pd.to_numeric(X_test_all['sleep_hours'], errors='coerce').astype(float)
        )
        med = X_train_all['study_hours_x_sleep_hours'].replace([np.inf, -np.inf], np.nan).median(skipna=True)
        if pd.isna(med):
            med = 0.0
        X_train_all['study_hours_x_sleep_hours'] = X_train_all['study_hours_x_sleep_hours'].replace([np.inf, -np.inf], np.nan).fillna(med)
        X_test_all['study_hours_x_sleep_hours'] = X_test_all['study_hours_x_sleep_hours'].replace([np.inf, -np.inf], np.nan).fillna(med)

    if all(col in X_train_all.columns for col in ['study_hours', 'sleep_quality']) and all(col in X_test_all.columns for col in ['study_hours', 'sleep_quality']):
        X_train_all['study_hours_x_sleep_quality'] = (
            pd.to_numeric(X_train_all['study_hours'], errors='coerce').astype(float)
            * pd.to_numeric(X_train_all['sleep_quality'], errors='coerce').astype(float)
        )
        X_test_all['study_hours_x_sleep_quality'] = (
            pd.to_numeric(X_test_all['study_hours'], errors='coerce').astype(float)
            * pd.to_numeric(X_test_all['sleep_quality'], errors='coerce').astype(float)
        )
        med = X_train_all['study_hours_x_sleep_quality'].replace([np.inf, -np.inf], np.nan).median(skipna=True)
        if pd.isna(med):
            med = 0.0
        X_train_all['study_hours_x_sleep_quality'] = X_train_all['study_hours_x_sleep_quality'].replace([np.inf, -np.inf], np.nan).fillna(med)
        X_test_all['study_hours_x_sleep_quality'] = X_test_all['study_hours_x_sleep_quality'].replace([np.inf, -np.inf], np.nan).fillna(med)

    y_all = pd.to_numeric(train_feat[target_col], errors='coerce').astype(float)
    valid_target = y_all.notna()

    train_feat = train_feat.loc[valid_target].reset_index(drop=True)
    X_train_all = X_train_all.loc[valid_target].reset_index(drop=True)
    y_all = y_all.loc[valid_target].reset_index(drop=True)
    test_feat = test_feat.reset_index(drop=True)
    X_test_all = X_test_all.reset_index(drop=True)

    ctx = {
        'train_raw': train_raw,
        'test_raw': test_raw,
        'data_dir': data_dir,
        'train_path': train_path,
        'test_path': test_path,
        'target_col': target_col,
        'id_col': id_col,
        'train_feat': train_feat,
        'test_feat': test_feat,
        'encoding_report_df': encoding_report_df,
        'X_train_all': X_train_all,
        'X_test_all': X_test_all,
        'y_all': y_all,
    }
    _LEGACY_CONTEXT_CACHE = ctx
    return ctx


def _to_num_series(series, col_name):
    if pd.api.types.is_object_dtype(series) or str(series.dtype).startswith('string'):
        if col_name in LEGACY_ORDINAL_MAPS:
            series = series.astype('string').str.strip().str.lower().map(LEGACY_ORDINAL_MAPS[col_name])
    return pd.to_numeric(series, errors='coerce').astype(float)


def _series_from_ctx(ctx, X_train, X_test, col_name):
    if col_name in X_train.columns and col_name in X_test.columns:
        return _to_num_series(X_train[col_name], col_name).reset_index(drop=True), _to_num_series(X_test[col_name], col_name).reset_index(drop=True)

    if col_name in ctx['train_feat'].columns and col_name in ctx['test_feat'].columns:
        return _to_num_series(ctx['train_feat'][col_name], col_name).reset_index(drop=True), _to_num_series(ctx['test_feat'][col_name], col_name).reset_index(drop=True)

    if col_name in ctx['train_raw'].columns and col_name in ctx['test_raw'].columns:
        tr = _to_num_series(ctx['train_raw'][col_name], col_name)
        te = _to_num_series(ctx['test_raw'][col_name], col_name)
        target_mask = pd.to_numeric(ctx['train_raw'][ctx['target_col']], errors='coerce').astype(float).notna()
        tr = tr.loc[target_mask].reset_index(drop=True)
        te = te.reset_index(drop=True)
        if len(tr) == len(X_train) and len(te) == len(X_test):
            return tr, te

    return None, None


def _fill_feature_frame(train_df, test_df, col_name):
    train_df[col_name] = train_df[col_name].replace([np.inf, -np.inf], np.nan)
    test_df[col_name] = test_df[col_name].replace([np.inf, -np.inf], np.nan)
    med = train_df[col_name].median(skipna=True)
    if pd.isna(med):
        med = 0.0
    train_df[col_name] = train_df[col_name].fillna(med)
    test_df[col_name] = test_df[col_name].fillna(med)


def _add_pair_from_ctx(ctx, train_df, test_df, feature_name, col_a, col_b, verbose=False):
    tr_a, te_a = _series_from_ctx(ctx, train_df, test_df, col_a)
    tr_b, te_b = _series_from_ctx(ctx, train_df, test_df, col_b)
    if tr_a is None or tr_b is None:
        if verbose:
            print(f'{feature_name} na pas pu etre ajoute : {col_a} veya {col_b} introuvable.')
        return False
    train_df[feature_name] = tr_a * tr_b
    test_df[feature_name] = te_a * te_b
    _fill_feature_frame(train_df, test_df, feature_name)
    if verbose:
        print(f'Nouvelle feature ajoutee : {feature_name} = {col_a} * {col_b}')
    return True


def _add_triple_from_ctx(ctx, train_df, test_df, feature_name, col_a, col_b, col_c, verbose=False):
    tr_a, te_a = _series_from_ctx(ctx, train_df, test_df, col_a)
    tr_b, te_b = _series_from_ctx(ctx, train_df, test_df, col_b)
    tr_c, te_c = _series_from_ctx(ctx, train_df, test_df, col_c)
    if tr_a is None or tr_b is None or tr_c is None:
        if verbose:
            print(f'{feature_name} na pas pu etre ajoute : {col_a}/{col_b}/{col_c} manquant.')
        return False
    train_df[feature_name] = tr_a * tr_b * tr_c
    test_df[feature_name] = te_a * te_b * te_c
    _fill_feature_frame(train_df, test_df, feature_name)
    if verbose:
        print(f'Nouvelle feature ajoutee : {feature_name} = {col_a} * {col_b} * {col_c}')
    return True


def build_legacy_feature_bundle(refresh=False, verbose=False):
    global _LEGACY_FEATURE_BUNDLE_CACHE
    if _LEGACY_FEATURE_BUNDLE_CACHE is not None and not refresh:
        return _LEGACY_FEATURE_BUNDLE_CACHE

    ctx = get_legacy_context(refresh=refresh, verbose=verbose)
    X_train = ctx['X_train_all'].copy()
    X_test = ctx['X_test_all'].copy()

    _add_pair_from_ctx(ctx, X_train, X_test, 'study_hours_x_sleep_hours', 'study_hours', 'sleep_hours', verbose=verbose)
    _add_pair_from_ctx(ctx, X_train, X_test, 'study_hours_x_sleep_quality', 'study_hours', 'sleep_quality', verbose=verbose)
    _add_pair_from_ctx(ctx, X_train, X_test, 'study_hours_x_facility_rating', 'study_hours', 'facility_rating', verbose=verbose)
    _add_pair_from_ctx(ctx, X_train, X_test, 'study_hours_x_class_attendance', 'study_hours', 'class_attendance', verbose=verbose)
    _add_triple_from_ctx(ctx, X_train, X_test, 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours', 'class_attendance', 'sleep_hours', verbose=verbose)

    study_train, study_test = _series_from_ctx(ctx, X_train, X_test, 'study_hours')
    if study_train is None or study_test is None:
        raise ValueError('La colonne study_hours est introuvable ou na pas pu etre generee.')

    if 'study_hours' in X_train.columns:
        X_train = X_train.drop(columns=['study_hours'])
    if 'study_hours' in X_test.columns:
        X_test = X_test.drop(columns=['study_hours'])

    X_train['study_hours_sq'] = study_train ** 2
    X_test['study_hours_sq'] = study_test ** 2
    _fill_feature_frame(X_train, X_test, 'study_hours_sq')

    if 'k3_f1' not in X_train.columns or 'k3_f1' not in X_test.columns:
        ok = _add_triple_from_ctx(ctx, X_train, X_test, 'k3_f1', 'facility_rating', 'sleep_hours', 'age', verbose=verbose)
        if not ok:
            raise ValueError('k3_f1 est introuvable et na pas pu etre regenere.')

    if 'k3_f2' not in X_train.columns or 'k3_f2' not in X_test.columns:
        ok = _add_triple_from_ctx(ctx, X_train, X_test, 'k3_f2', 'sleep_quality', 'facility_rating', 'internet_access', verbose=verbose)
        if not ok:
            raise ValueError('k3_f2 est introuvable et na pas pu etre regenere.')

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    med = X_train.median(numeric_only=True)
    X_train = X_train.fillna(med)
    X_test = X_test.fillna(med)

    bundle = {
        'ctx': ctx,
        'X_train': X_train,
        'X_test': X_test,
        'y': ctx['y_all'].copy().reset_index(drop=True),
        'feature_cols': X_train.columns.tolist(),
        'target_col': ctx['target_col'],
        'test': ctx['test_raw'],
        'train': ctx['train_raw'],
    }
    _LEGACY_FEATURE_BUNDLE_CACHE = bundle
    return bundle


def build_simple_trim_bundle(refresh=False, verbose=False):
    global _SIMPLE_TRIM_BUNDLE_CACHE
    if _SIMPLE_TRIM_BUNDLE_CACHE is not None and not refresh:
        return _SIMPLE_TRIM_BUNDLE_CACHE

    ctx = get_legacy_context(refresh=refresh, verbose=verbose)
    simple_trim_cols = ['age', 'study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating']
    missing_cols = [c for c in simple_trim_cols if c not in ctx['train_feat'].columns or c not in ctx['test_feat'].columns]
    if missing_cols:
        raise ValueError(f'Colonnes manquantes pour Simple Trim : {missing_cols}')

    X_train = ctx['train_feat'][simple_trim_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    X_test = ctx['test_feat'][simple_trim_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    y = ctx['y_all'].copy().reset_index(drop=True)

    X_train['study_hours_sq'] = pd.to_numeric(X_train['study_hours'], errors='coerce').astype(float) ** 2
    X_test['study_hours_sq'] = pd.to_numeric(X_test['study_hours'], errors='coerce').astype(float) ** 2
    X_train['study_hours_x_class_attendance'] = pd.to_numeric(X_train['study_hours'], errors='coerce').astype(float) * pd.to_numeric(X_train['class_attendance'], errors='coerce').astype(float)
    X_test['study_hours_x_class_attendance'] = pd.to_numeric(X_test['study_hours'], errors='coerce').astype(float) * pd.to_numeric(X_test['class_attendance'], errors='coerce').astype(float)
    X_train['study_hours_x_sleep_hours'] = pd.to_numeric(X_train['study_hours'], errors='coerce').astype(float) * pd.to_numeric(X_train['sleep_hours'], errors='coerce').astype(float)
    X_test['study_hours_x_sleep_hours'] = pd.to_numeric(X_test['study_hours'], errors='coerce').astype(float) * pd.to_numeric(X_test['sleep_hours'], errors='coerce').astype(float)

    med = X_train.median(numeric_only=True)
    X_train = X_train.fillna(med)
    X_test = X_test.fillna(med)

    bundle = {
        'ctx': ctx,
        'X_train': X_train.reset_index(drop=True),
        'X_test': X_test.reset_index(drop=True),
        'y': y.reset_index(drop=True),
        'feature_cols': X_train.columns.tolist(),
        'target_col': ctx['target_col'],
        'test': ctx['test_raw'],
    }
    _SIMPLE_TRIM_BUNDLE_CACHE = bundle
    return bundle


def _get_target_encoder_backend():
    return require_target_encoder_backend()


def get_course_style_context(refresh=False, verbose=False):
    global _COURSE_STYLE_CONTEXT_CACHE
    if _COURSE_STYLE_CONTEXT_CACHE is not None and not refresh:
        return _COURSE_STYLE_CONTEXT_CACHE

    train, test, data_dir, train_path, test_path = load_local_train_test(verbose=verbose)
    te_backend, TargetEncoder = _get_target_encoder_backend()

    TARGET = 'exam_score'
    ID_COL = 'id' if 'id' in train.columns and 'id' in test.columns else None
    if TARGET not in train.columns:
        raise ValueError(f'Colonne cible introuvable: {TARGET}')

    train_stack = train.copy()
    test_stack = test.copy()
    y = pd.to_numeric(train_stack[TARGET], errors='coerce').astype(float)
    valid_target = y.notna()
    train_stack = train_stack.loc[valid_target].reset_index(drop=True)
    y = y.loc[valid_target].reset_index(drop=True)
    test_ids = test_stack[ID_COL].copy() if ID_COL is not None else pd.Series(np.arange(len(test_stack)))

    X = train_stack.drop(columns=[c for c in [ID_COL, TARGET] if c is not None])
    X_test = test_stack.drop(columns=[c for c in [ID_COL] if c is not None])

    NUMS = ['age', 'study_hours', 'class_attendance', 'sleep_hours']
    CATS = ['gender', 'course', 'internet_access', 'study_method', 'facility_rating', 'exam_difficulty', 'sleep_quality']

    for col in NUMS:
        X[col] = pd.to_numeric(X[col], errors='coerce').astype(float)
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce').astype(float)

    for col in CATS:
        X[col] = X[col].astype('string').str.strip().str.lower()
        X_test[col] = X_test[col].astype('string').str.strip().str.lower()

    for df in [X, X_test]:
        for p in [12, 14, 20]:
            df[f'_study_hours_sin_{p}'] = np.sin(2 * np.pi * df['study_hours'] / p)
            df[f'_class_attendance_sin_{p}'] = np.sin(2 * np.pi * df['class_attendance'] / p)
        for c in NUMS:
            df[f'{c}_sq'] = df[c] ** 2
        df['study_attendance_inter'] = df['study_hours'] * df['class_attendance']
        df['study_sleep_ratio'] = df['study_hours'] / (df['sleep_hours'] + 1e-5)
        df['magic_formula_1'] = 5.905 * df['study_hours'] + 0.345 * df['class_attendance'] + 1.423 * df['sleep_hours'] + 4.7819
        df['magic_formula_2'] = (
            6 * df['study_hours'] + 0.35 * df['class_attendance'] + 1.5 * df['sleep_hours']
            + 5 * (df['sleep_quality'] == 'good').astype(float)
            - 5 * (df['sleep_quality'] == 'poor').astype(float)
            + 10 * (df['study_method'] == 'coaching').astype(float)
            + 5 * (df['study_method'] == 'mixed').astype(float)
            + 2 * (df['study_method'] == 'group study').astype(float)
            + 1 * (df['study_method'] == 'online videos').astype(float)
            + 4 * (df['facility_rating'] == 'high').astype(float)
            - 4 * (df['facility_rating'] == 'low').astype(float)
        )

    pair_columns = []
    selected_pairs = [
        ('study_hours', 'gender'),
        ('study_hours', 'internet_access'),
        ('study_hours', 'sleep_quality'),
        ('study_hours', 'study_method'),
    ]
    for col1, col2 in selected_pairs:
        name = f'{col1}_{col2}'
        pair_columns.append(name)
        X[name] = X[col1].astype(str) + '_' + X[col2].astype(str)
        X_test[name] = X_test[col1].astype(str) + '_' + X_test[col2].astype(str)

    ALL_CATS = CATS + pair_columns

    if te_backend == 'sklearn':
        try:
            te = TargetEncoder(cv=5, smooth='auto', random_state=42)
        except TypeError:
            te = TargetEncoder(cv=5, smooth='auto')
        X_te = np.asarray(te.fit_transform(X[ALL_CATS], y))
        X_test_te = np.asarray(te.transform(X_test[ALL_CATS]))
    else:
        te = TargetEncoder(cols=ALL_CATS)
        X_te = te.fit_transform(X[ALL_CATS], y).to_numpy()
        X_test_te = te.transform(X_test[ALL_CATS]).to_numpy()

    for i, col in enumerate(ALL_CATS):
        X[f'TE_{col}'] = X_te[:, i]
        X_test[f'TE_{col}'] = X_test_te[:, i]

    try:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=np.nan, encoded_missing_value=np.nan)
    except TypeError:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    X_cat = pd.DataFrame(oe.fit_transform(X[ALL_CATS]), columns=ALL_CATS, index=X.index)
    X_test_cat = pd.DataFrame(oe.transform(X_test[ALL_CATS]), columns=ALL_CATS, index=X_test.index)
    X_cat = X_cat.mask(X_cat < 0, np.nan)
    X_test_cat = X_test_cat.mask(X_test_cat < 0, np.nan)
    X[ALL_CATS] = X_cat
    X_test[ALL_CATS] = X_test_cat

    X = X.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    X_test = X_test.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    train_medians = X.median(numeric_only=True)
    X = X.fillna(train_medians)
    X_test = X_test.fillna(train_medians)

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    sfs_sample_n = min(15000, len(X_scaled))
    sfs_feature_n = min(15, X_scaled.shape[1])
    sfs = SequentialFeatureSelector(
        Ridge(alpha=1.0),
        n_features_to_select=sfs_feature_n,
        direction='forward',
        cv=3,
        n_jobs=-1,
    )
    sfs.fit(X_scaled.iloc[:sfs_sample_n], y.iloc[:sfs_sample_n])

    selected_sfs_cols = X_scaled.columns[sfs.get_support()].tolist()
    X_scaled_sfs = sfs.transform(X_scaled)
    X_test_scaled_sfs = sfs.transform(X_test_scaled)

    ctx = {
        'data_dir': data_dir,
        'train_path': train_path,
        'test_path': test_path,
        'train': train,
        'test': test,
        'TARGET': TARGET,
        'ID_COL': ID_COL,
        'test_ids': test_ids,
        'y': y,
        'X': X,
        'X_test': X_test,
        'X_scaled': X_scaled,
        'X_test_scaled': X_test_scaled,
        'X_scaled_sfs': X_scaled_sfs,
        'X_test_scaled_sfs': X_test_scaled_sfs,
        'selected_sfs_cols': selected_sfs_cols,
        'NUMS': NUMS,
        'CATS': CATS,
    }
    _COURSE_STYLE_CONTEXT_CACHE = ctx

    if verbose:
        print('Features SFS :', selected_sfs_cols)
        print(f'Taille SFS : X_train={X_scaled_sfs.shape}, X_test={X_test_scaled_sfs.shape}')
    return ctx

In [ ]:
# 5.1 Regression Polynomiale (Legacy)

ctx = get_legacy_context(verbose=False)
poly_feature_cols = selected_base_features if 'selected_base_features' in globals() and selected_base_features else ctx['X_train_all'].columns.tolist()[:6]

X_poly_train = ctx['X_train_all'][poly_feature_cols].copy()
X_poly_test = ctx['X_test_all'][poly_feature_cols].copy()
y_poly = ctx['y_all'].copy().reset_index(drop=True)

POLY_MODEL_SAMPLE_N = min(120000, len(X_poly_train))
if len(X_poly_train) > POLY_MODEL_SAMPLE_N:
    sample_idx = X_poly_train.sample(n=POLY_MODEL_SAMPLE_N, random_state=42).index
    X_poly_train = X_poly_train.loc[sample_idx].reset_index(drop=True)
    y_poly = y_poly.loc[sample_idx].reset_index(drop=True)

kf = KFold(n_splits=2, shuffle=True, random_state=42)
oof = np.zeros(len(X_poly_train), dtype=float)
fold_rows = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_poly_train, y_poly), start=1):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('linreg', LinearRegression()),
    ])
    model.fit(X_poly_train.iloc[tr_idx], y_poly.iloc[tr_idx])
    preds = np.clip(model.predict(X_poly_train.iloc[va_idx]), 0.0, 100.0)
    oof[va_idx] = preds
    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_poly.iloc[va_idx], preds))),
        'mae': float(mean_absolute_error(y_poly.iloc[va_idx], preds)),
    })

poly_cv_rmse = float(np.sqrt(mean_squared_error(y_poly, oof)))
poly_cv_mae = float(mean_absolute_error(y_poly, oof))
poly_fold_df = pd.DataFrame(fold_rows)

final_poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('linreg', LinearRegression()),
])
final_poly_model.fit(X_poly_train, y_poly)
poly_test_pred = np.clip(final_poly_model.predict(X_poly_test), 0.0, 100.0)

poly_out_dir = Path('notebooks/reports/section5_polynomial_legacy')
poly_out_dir.mkdir(parents=True, exist_ok=True)
poly_submission = pd.DataFrame({ctx['target_col']: poly_test_pred})
if 'id' in ctx['test_raw'].columns:
    poly_submission = pd.DataFrame({'id': ctx['test_raw']['id'], ctx['target_col']: poly_test_pred})
poly_submission.to_csv(poly_out_dir / 'submission_polynomial_legacy.csv', index=False)
poly_fold_df.to_csv(poly_out_dir / 'polynomial_legacy_folds.csv', index=False)

print(f'Polynomial CV RMSE: {poly_cv_rmse:.6f}')
print(f'Polynomial CV MAE : {poly_cv_mae:.6f}')
display(poly_fold_df)
print('Enregistre :', poly_out_dir.resolve())

Polynomial CV RMSE: 9.045502
Polynomial CV MAE : 7.229642


,fold,rmse,mae
0,1,9.048221,7.230678
1,2,9.042782,7.228605


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_polynomial_legacy


In [ ]:
# 5.2 XGBoost (Legacy)

XGBRegressor = require_xgboost_regressor()
bundle = build_legacy_feature_bundle(verbose=True)
X_xgb_train = bundle['X_train'].copy()
X_xgb_test = bundle['X_test'].copy()
y_xgb = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])

XGB_SAMPLE_N = min(630000, len(X_xgb_train))
if len(X_xgb_train) > XGB_SAMPLE_N:
    sample_idx = X_xgb_train.sample(n=XGB_SAMPLE_N, random_state=42).index
    X_xgb_train = X_xgb_train.loc[sample_idx].reset_index(drop=True)
    y_xgb = y_xgb.loc[sample_idx].reset_index(drop=True)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_child_weight': 25,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 5.0,
    'gamma': 1.0,
    'random_state': 42,
    'n_jobs': -1,
}

early_stopping_rounds = 50
kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_xgb_train), dtype=float)
fold_rows = []
best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_xgb_train, y_xgb), start=1):
    X_tr = X_xgb_train.iloc[tr_idx]
    y_tr = y_xgb.iloc[tr_idx]
    X_va = X_xgb_train.iloc[va_idx]
    y_va = y_xgb.iloc[va_idx]

    model = XGBRegressor(**params, early_stopping_rounds=early_stopping_rounds)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        best_rounds.append(int(best_iter) + 1)

    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

xgb_cv_rmse = float(np.sqrt(mean_squared_error(y_xgb, oof)))
xgb_cv_mae = float(mean_absolute_error(y_xgb, oof))
xgb_fold_df = pd.DataFrame(fold_rows)

final_n_estimators = int(np.median(best_rounds)) if best_rounds else params['n_estimators']
final_params = params.copy()
final_params['n_estimators'] = final_n_estimators
print('n_estimators final :', final_n_estimators)

final_xgb_model = XGBRegressor(**final_params)
final_xgb_model.fit(X_xgb_train, y_xgb)
xgb_test_pred = np.clip(final_xgb_model.predict(X_xgb_test), 0.0, 100.0)

xgb_out_dir = Path('notebooks/reports/section5_xgb_legacy')
xgb_out_dir.mkdir(parents=True, exist_ok=True)
xgb_submission = pd.DataFrame({bundle['target_col']: xgb_test_pred})
if 'id' in bundle['test'].columns:
    xgb_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: xgb_test_pred})
xgb_submission.to_csv(xgb_out_dir / 'submission_xgb_legacy_all_replace_study_hours_with_sq.csv', index=False)
xgb_fold_df.to_csv(xgb_out_dir / 'xgb_legacy_all_replace_study_hours_with_sq_folds.csv', index=False)

print(f'XGB CV RMSE: {xgb_cv_rmse:.6f}')
print(f'XGB CV MAE : {xgb_cv_mae:.6f}')
display(xgb_fold_df)
print('Enregistre :', xgb_out_dir.resolve())

Nouvelle feature ajoutee : study_hours_x_sleep_hours = study_hours * sleep_hours
Nouvelle feature ajoutee : study_hours_x_sleep_quality = study_hours * sleep_quality
Nouvelle feature ajoutee : study_hours_x_facility_rating = study_hours * facility_rating
Nouvelle feature ajoutee : study_hours_x_class_attendance = study_hours * class_attendance
Nouvelle feature ajoutee : study_hours_x_class_attendance_x_sleep_hours = study_hours * class_attendance * sleep_hours
Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
n_estimators final : 1000
XGB CV RMSE: 8.762178
XGB CV MAE : 6.981746


,fold,rmse,mae
0,1,8.747897,6.975074
1,2,8.759001,6.982467
2,3,8.779607,6.987698


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_legacy


In [ ]:
# 5.3 XGBoost (Selection Simple)
XGBRegressor = require_xgboost_regressor()

bundle = build_simple_trim_bundle(verbose=False)
X_simple_train = bundle['X_train'].copy()
X_simple_test = bundle['X_test'].copy()
y_simple = bundle['y'].copy().reset_index(drop=True)

print('Features Selection Simple :', bundle['feature_cols'])
print(f'Taille Selection Simple : X_train={X_simple_train.shape}, X_test={X_simple_test.shape}')

XGB_SIMPLE_SAMPLE_N = min(630000, len(X_simple_train))
if len(X_simple_train) > XGB_SIMPLE_SAMPLE_N:
    sample_idx = X_simple_train.sample(n=XGB_SIMPLE_SAMPLE_N, random_state=42).index
    X_simple_train = X_simple_train.loc[sample_idx].reset_index(drop=True)
    y_simple = y_simple.loc[sample_idx].reset_index(drop=True)

simple_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 3000,
    'learning_rate': 0.03,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.9,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.0,
    'reg_lambda': 4.0,
    'gamma': 0.0,
    'random_state': 42,
    'n_jobs': -1,
}

simple_early_stopping_rounds = 100
simple_kf = KFold(n_splits=3, shuffle=True, random_state=42)
simple_oof = np.zeros(len(X_simple_train), dtype=float)
simple_fold_rows = []
simple_best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(simple_kf.split(X_simple_train, y_simple), start=1):
    X_tr = X_simple_train.iloc[tr_idx]
    y_tr = y_simple.iloc[tr_idx]
    X_va = X_simple_train.iloc[va_idx]
    y_va = y_simple.iloc[va_idx]

    model = XGBRegressor(**simple_params, early_stopping_rounds=simple_early_stopping_rounds)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    simple_oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        simple_best_rounds.append(int(best_iter) + 1)

    simple_fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

simple_cv_rmse = float(np.sqrt(mean_squared_error(y_simple, simple_oof)))
simple_cv_mae = float(mean_absolute_error(y_simple, simple_oof))
simple_fold_df = pd.DataFrame(simple_fold_rows)

simple_final_n_estimators = int(np.median(simple_best_rounds)) if simple_best_rounds else simple_params['n_estimators']
simple_final_params = simple_params.copy()
simple_final_params['n_estimators'] = simple_final_n_estimators
print('n_estimators final (Selection Simple) :', simple_final_n_estimators)

simple_final_model = XGBRegressor(**simple_final_params)
simple_final_model.fit(X_simple_train, y_simple)
simple_test_pred = np.clip(simple_final_model.predict(X_simple_test), 0.0, 100.0)

simple_out_dir = Path('notebooks/reports/section5_xgb_simple_trim')
simple_out_dir.mkdir(parents=True, exist_ok=True)
simple_submission = pd.DataFrame({bundle['target_col']: simple_test_pred})
if 'id' in bundle['test'].columns:
    simple_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: simple_test_pred})
simple_submission.to_csv(simple_out_dir / 'submission_xgb_simple_trim.csv', index=False)
simple_fold_df.to_csv(simple_out_dir / 'xgb_simple_trim_folds.csv', index=False)

print(f'Simple Trim XGB CV RMSE: {simple_cv_rmse:.6f}')
print(f'Simple Trim XGB CV MAE : {simple_cv_mae:.6f}')
display(simple_fold_df)
print('Enregistre :', simple_out_dir.resolve())

Features Selection Simple : ['age', 'study_hours', 'class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'facility_rating', 'study_hours_sq', 'study_hours_x_class_attendance', 'study_hours_x_sleep_hours']
Taille Selection Simple : X_train=(630000, 10), X_test=(270000, 10)
n_estimators final (Selection Simple) : 3000
Simple Trim XGB CV RMSE: 8.764463
Simple Trim XGB CV MAE : 6.988846


,fold,rmse,mae
0,1,8.751737,6.982890
1,2,8.760148,6.989879
2,3,8.781478,6.993769


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_simple_trim


In [ ]:
# 5.4 XGBoost (Sauvegarde Legacy)

XGBRegressor = require_xgboost_regressor()
bundle = build_legacy_feature_bundle(verbose=True)
X_xgb_train = bundle['X_train'].copy()
X_xgb_test = bundle['X_test'].copy()
y_xgb = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])

XGB_SAMPLE_N = min(630000, len(X_xgb_train))
if len(X_xgb_train) > XGB_SAMPLE_N:
    sample_idx = X_xgb_train.sample(n=XGB_SAMPLE_N, random_state=42).index
    X_xgb_train = X_xgb_train.loc[sample_idx].reset_index(drop=True)
    y_xgb = y_xgb.loc[sample_idx].reset_index(drop=True)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 1000,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_child_weight': 25,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 5.0,
    'gamma': 1.0,
    'random_state': 42,
    'n_jobs': -1,
}

early_stopping_rounds = 50
kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_xgb_train), dtype=float)
fold_rows = []
best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_xgb_train, y_xgb), start=1):
    X_tr = X_xgb_train.iloc[tr_idx]
    y_tr = y_xgb.iloc[tr_idx]
    X_va = X_xgb_train.iloc[va_idx]
    y_va = y_xgb.iloc[va_idx]

    model = XGBRegressor(**params, early_stopping_rounds=early_stopping_rounds)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        best_rounds.append(int(best_iter) + 1)

    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

xgb_cv_rmse = float(np.sqrt(mean_squared_error(y_xgb, oof)))
xgb_cv_mae = float(mean_absolute_error(y_xgb, oof))
xgb_fold_df = pd.DataFrame(fold_rows)

final_n_estimators = int(np.median(best_rounds)) if best_rounds else params['n_estimators']
final_params = params.copy()
final_params['n_estimators'] = final_n_estimators
print('n_estimators final :', final_n_estimators)

final_xgb_model = XGBRegressor(**final_params)
final_xgb_model.fit(X_xgb_train, y_xgb)
xgb_test_pred = np.clip(final_xgb_model.predict(X_xgb_test), 0.0, 100.0)

xgb_out_dir = Path('notebooks/reports/section5_xgb_legacy_backup')
xgb_out_dir.mkdir(parents=True, exist_ok=True)
xgb_submission = pd.DataFrame({bundle['target_col']: xgb_test_pred})
if 'id' in bundle['test'].columns:
    xgb_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: xgb_test_pred})
xgb_submission.to_csv(xgb_out_dir / 'submission_xgb_legacy_backup.csv', index=False)
xgb_fold_df.to_csv(xgb_out_dir / 'xgb_legacy_backup_folds.csv', index=False)

print(f'XGB CV RMSE: {xgb_cv_rmse:.6f}')
print(f'XGB CV MAE : {xgb_cv_mae:.6f}')
display(xgb_fold_df)
print('Enregistre :', xgb_out_dir.resolve())

Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
n_estimators final : 1000
XGB CV RMSE: 8.762178
XGB CV MAE : 6.981746


,fold,rmse,mae
0,1,8.747897,6.975074
1,2,8.759001,6.982467
2,3,8.779607,6.987698


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_legacy_backup


In [ ]:
# 5.5 CatBoost (Features Legacy)

CatBoostRegressor = require_catboost_regressor()
bundle = build_legacy_feature_bundle(verbose=True)
X_cb_train = bundle['X_train'].copy()
X_cb_test = bundle['X_test'].copy()
y_cb = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])

params = {
    'loss_function': 'RMSE',
    'eval_metric': 'RMSE',
    'iterations': 5000,
    'learning_rate': 0.01,
    'depth': 5,
    'l2_leaf_reg': 5.0,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
    'rsm': 0.7,
    'random_seed': 42,
    'verbose': False,
}

early_stopping_rounds = 50
kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_cb_train), dtype=float)
fold_rows = []
best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_cb_train, y_cb), start=1):
    X_tr = X_cb_train.iloc[tr_idx]
    X_va = X_cb_train.iloc[va_idx]
    y_tr = y_cb.iloc[tr_idx]
    y_va = y_cb.iloc[va_idx]

    model = CatBoostRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, early_stopping_rounds=early_stopping_rounds, verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration_', None)
    if best_iter is not None and best_iter > 0:
        best_rounds.append(int(best_iter) + 1)

    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

cb_cv_rmse = float(np.sqrt(mean_squared_error(y_cb, oof)))
cb_cv_mae = float(mean_absolute_error(y_cb, oof))
cb_fold_df = pd.DataFrame(fold_rows)

final_iterations = int(np.median(best_rounds)) if best_rounds else params['iterations']
final_params = params.copy()
final_params['iterations'] = final_iterations
final_model = CatBoostRegressor(**final_params)
final_model.fit(X_cb_train, y_cb, verbose=False)
cb_test_pred = np.clip(final_model.predict(X_cb_test), 0.0, 100.0)

cb_out_dir = Path('notebooks/reports/section5_catboost_legacy_features')
cb_out_dir.mkdir(parents=True, exist_ok=True)
cb_submission = pd.DataFrame({bundle['target_col']: cb_test_pred})
if 'id' in bundle['test'].columns:
    cb_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: cb_test_pred})
cb_submission.to_csv(cb_out_dir / 'submission_catboost_legacy_features.csv', index=False)
cb_fold_df.to_csv(cb_out_dir / 'catboost_legacy_features_folds.csv', index=False)

print(f'CatBoost CV RMSE: {cb_cv_rmse:.6f}')
print(f'CatBoost CV MAE : {cb_cv_mae:.6f}')
print(f'Iterations finales : {final_iterations}')
display(cb_fold_df)
print('Enregistre :', cb_out_dir.resolve())

Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
CatBoost CV RMSE: 8.805352
CatBoost CV MAE : 7.028816
Iterations finales : 5000


,fold,rmse,mae
0,1,8.794558,7.024326
1,2,8.803644,7.031741
2,3,8.817838,7.030382


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_catboost_legacy_features


In [ ]:
# 5.6 Regression Polynomiale (Features Legacy)

bundle = build_legacy_feature_bundle(verbose=True)
X_poly_train = bundle['X_train'].copy()
X_poly_test = bundle['X_test'].copy()
y_poly = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])

POLY_SAMPLE_N = min(180000, len(X_poly_train))
if len(X_poly_train) > POLY_SAMPLE_N:
    sample_idx = X_poly_train.sample(n=POLY_SAMPLE_N, random_state=42).index
    X_poly_train = X_poly_train.loc[sample_idx].reset_index(drop=True)
    y_poly = y_poly.loc[sample_idx].reset_index(drop=True)

poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
    ('ridge', Ridge(alpha=5.0)),
])

kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_poly_train), dtype=float)
fold_rows = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_poly_train, y_poly), start=1):
    X_tr = X_poly_train.iloc[tr_idx]
    X_va = X_poly_train.iloc[va_idx]
    y_tr = y_poly.iloc[tr_idx]
    y_va = y_poly.iloc[va_idx]

    poly_model.fit(X_tr, y_tr)
    preds = np.clip(poly_model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds
    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

poly_cv_rmse = float(np.sqrt(mean_squared_error(y_poly, oof)))
poly_cv_mae = float(mean_absolute_error(y_poly, oof))
poly_fold_df = pd.DataFrame(fold_rows)

final_poly_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scale', StandardScaler()),
    ('ridge', Ridge(alpha=5.0)),
])
final_poly_model.fit(X_poly_train, y_poly)
poly_test_pred = np.clip(final_poly_model.predict(X_poly_test), 0.0, 100.0)

poly_out_dir = Path('notebooks/reports/section5_polynomial_legacy_features')
poly_out_dir.mkdir(parents=True, exist_ok=True)
poly_submission = pd.DataFrame({bundle['target_col']: poly_test_pred})
if 'id' in bundle['test'].columns:
    poly_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: poly_test_pred})
poly_submission.to_csv(poly_out_dir / 'submission_polynomial_legacy_features.csv', index=False)
poly_fold_df.to_csv(poly_out_dir / 'polynomial_legacy_features_folds.csv', index=False)

print(f'Polynomial CV RMSE: {poly_cv_rmse:.6f}')
print(f'Polynomial CV MAE : {poly_cv_mae:.6f}')
display(poly_fold_df)
print('Enregistre :', poly_out_dir.resolve())

Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
Polynomial CV RMSE: 9.023082
Polynomial CV MAE : 7.222413


,fold,rmse,mae
0,1,9.023302,7.223796
1,2,9.016080,7.210835
2,3,9.029859,7.232609


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_polynomial_legacy_features


In [ ]:
# 5.7 Stacking Style Cours (HistGB + Ridge + LinearSVR)

ctx = get_course_style_context(verbose=True)

X = ctx['X']
X_test = ctx['X_test']
y = ctx['y'].copy().reset_index(drop=True)
X_scaled_sfs = ctx['X_scaled_sfs']
X_test_scaled_sfs = ctx['X_test_scaled_sfs']
selected_sfs_cols = ctx['selected_sfs_cols']
TARGET = ctx['TARGET']
ID_COL = ctx['ID_COL']
test_ids = ctx['test_ids']
CATS = ctx['CATS']

print(f'SFS a reduit {ctx["X"].shape[1]} features a {len(selected_sfs_cols)} features.')
print('SFS features:', selected_sfs_cols)

N_SPLITS = 10
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_gbm = np.zeros(len(X), dtype=float)
oof_ridge = np.zeros(len(X), dtype=float)
oof_svr = np.zeros(len(X), dtype=float)
test_preds_gbm = np.zeros(len(X_test), dtype=float)
test_preds_ridge = np.zeros(len(X_test), dtype=float)
test_preds_svr = np.zeros(len(X_test), dtype=float)
cat_indices = [X.columns.get_loc(c) for c in CATS if c in X.columns]
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
    print(f'Fold {fold}/{N_SPLITS}')
    X_tr = X.iloc[train_idx]
    X_va = X.iloc[val_idx]
    Xsfs_tr = X_scaled_sfs[train_idx]
    Xsfs_va = X_scaled_sfs[val_idx]
    y_tr = y.iloc[train_idx]
    y_va = y.iloc[val_idx]

    gbm = HistGradientBoostingRegressor(
        categorical_features=cat_indices,
        max_iter=600,
        learning_rate=0.04,
        l2_regularization=1.0,
        random_state=42,
    )
    gbm.fit(X_tr, y_tr)
    gbm_val = gbm.predict(X_va)
    oof_gbm[val_idx] = gbm_val
    test_preds_gbm += gbm.predict(X_test) / N_SPLITS

    ridge = Ridge(alpha=10.0)
    ridge.fit(Xsfs_tr, y_tr)
    ridge_val = ridge.predict(Xsfs_va)
    oof_ridge[val_idx] = ridge_val
    test_preds_ridge += ridge.predict(X_test_scaled_sfs) / N_SPLITS

    svr = LinearSVR(C=0.1, epsilon=0.05, dual=False, loss='squared_epsilon_insensitive', random_state=42, max_iter=5000)
    svr.fit(Xsfs_tr, y_tr)
    svr_val = svr.predict(Xsfs_va)
    oof_svr[val_idx] = svr_val
    test_preds_svr += svr.predict(X_test_scaled_sfs) / N_SPLITS

    fold_rows.append({
        'fold': fold,
        'gbm_rmse': float(np.sqrt(mean_squared_error(y_va, gbm_val))),
        'ridge_rmse': float(np.sqrt(mean_squared_error(y_va, ridge_val))),
        'svr_rmse': float(np.sqrt(mean_squared_error(y_va, svr_val))),
    })

print(f'RMSE GBM niveau 1 :   {np.sqrt(mean_squared_error(y, oof_gbm)):.6f}')
print(f'RMSE Ridge niveau 1 : {np.sqrt(mean_squared_error(y, oof_ridge)):.6f}')
print(f'RMSE SVR niveau 1 :   {np.sqrt(mean_squared_error(y, oof_svr)):.6f}')

OOF_train = np.column_stack((oof_gbm, oof_ridge, oof_svr))
OOF_test = np.column_stack((test_preds_gbm, test_preds_ridge, test_preds_svr))
meta_model = Ridge(alpha=1.0)
meta_model.fit(OOF_train, y)

print(f'Poids du blend : GBM={meta_model.coef_[0]:.4f}, Ridge={meta_model.coef_[1]:.4f}, SVR={meta_model.coef_[2]:.4f}')

final_preds = np.clip(meta_model.predict(OOF_test), 19.6, 100.0)
final_oof = np.clip(meta_model.predict(OOF_train), 19.6, 100.0)
stacked_rmse = float(np.sqrt(mean_squared_error(y, final_oof)))
print(f'RMSE CV finale du stacking : {stacked_rmse:.6f}')

out_dir = Path('notebooks/reports/section5_course_style_stacking')
out_dir.mkdir(parents=True, exist_ok=True)
submission = pd.DataFrame({TARGET: final_preds})
if ID_COL is not None:
    submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_preds})
submission.to_csv(out_dir / 'submission_course_style_stacking.csv', index=False)
pd.DataFrame(fold_rows).to_csv(out_dir / 'course_style_stacking_folds.csv', index=False)
pd.DataFrame({
    'metric': ['gbm_rmse', 'ridge_rmse', 'svr_rmse', 'stacked_rmse'],
    'value': [
        float(np.sqrt(mean_squared_error(y, oof_gbm))),
        float(np.sqrt(mean_squared_error(y, oof_ridge))),
        float(np.sqrt(mean_squared_error(y, oof_svr))),
        stacked_rmse,
    ],
}).to_csv(out_dir / 'course_style_stacking_summary.csv', index=False)

print('Enregistre :', out_dir.resolve())

DATA_DIR   : /Users/selimkusursuz/Desktop/ML 2026/ML2026/donnees
Train path : /Users/selimkusursuz/Desktop/ML 2026/ML2026/donnees/train.csv
Test path  : /Users/selimkusursuz/Desktop/ML 2026/ML2026/donnees/test.csv
Forme train: (630000, 13)
Forme test : (270000, 12)
Features SFS : ['study_hours', 'class_attendance', 'sleep_quality', 'exam_difficulty', '_class_attendance_sin_12', '_class_attendance_sin_14', 'class_attendance_sq', 'study_sleep_ratio', 'magic_formula_2', 'study_hours_internet_access', 'TE_course', 'TE_study_method', 'TE_exam_difficulty', 'TE_study_hours_internet_access', 'TE_study_hours_study_method']
Taille SFS : X_train=(630000, 15), X_test=(270000, 15)
SFS a reduit 40 features a 15 features.
SFS features: ['study_hours', 'class_attendance', 'sleep_quality', 'exam_difficulty', '_class_attendance_sin_12', '_class_attendance_sin_14', 'class_attendance_sq', 'study_sleep_ratio', 'magic_formula_2', 'study_hours_internet_access', 'TE_course', 'TE_study_method', 'TE_exam_diffic

In [ ]:
# 5.8 Ridge + LinearSVR (Features Legacy)

bundle = build_legacy_feature_bundle(verbose=True)
X_lin_train = bundle['X_train'].copy()
X_lin_test = bundle['X_test'].copy()
y_lin = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])
print(f'Taille du modele lineaire : X_train={X_lin_train.shape}, X_test={X_lin_test.shape}')

scaler = StandardScaler()
X_lin_train_scaled = scaler.fit_transform(X_lin_train)
X_lin_test_scaled = scaler.transform(X_lin_test)

kf = KFold(n_splits=3, shuffle=True, random_state=42)
ridge_oof = np.zeros(len(X_lin_train_scaled), dtype=float)
svr_oof = np.zeros(len(X_lin_train_scaled), dtype=float)
ridge_test_pred = np.zeros(len(X_lin_test_scaled), dtype=float)
svr_test_pred = np.zeros(len(X_lin_test_scaled), dtype=float)
fold_rows = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_lin_train_scaled, y_lin), start=1):
    X_tr = X_lin_train_scaled[tr_idx]
    X_va = X_lin_train_scaled[va_idx]
    y_tr = y_lin.iloc[tr_idx]
    y_va = y_lin.iloc[va_idx]

    ridge_model = Ridge(alpha=10.0)
    ridge_model.fit(X_tr, y_tr)
    ridge_preds = np.clip(ridge_model.predict(X_va), 0.0, 100.0)
    ridge_oof[va_idx] = ridge_preds
    ridge_test_pred += np.clip(ridge_model.predict(X_lin_test_scaled), 0.0, 100.0) / kf.n_splits

    svr_model = LinearSVR(C=0.1, epsilon=0.05, dual=False, loss='squared_epsilon_insensitive', random_state=42, max_iter=5000)
    svr_model.fit(X_tr, y_tr)
    svr_preds = np.clip(svr_model.predict(X_va), 0.0, 100.0)
    svr_oof[va_idx] = svr_preds
    svr_test_pred += np.clip(svr_model.predict(X_lin_test_scaled), 0.0, 100.0) / kf.n_splits

    fold_rows.append({
        'fold': fold_idx,
        'ridge_rmse': float(np.sqrt(mean_squared_error(y_va, ridge_preds))),
        'ridge_mae': float(mean_absolute_error(y_va, ridge_preds)),
        'linear_svr_rmse': float(np.sqrt(mean_squared_error(y_va, svr_preds))),
        'linear_svr_mae': float(mean_absolute_error(y_va, svr_preds)),
    })

ridge_cv_rmse = float(np.sqrt(mean_squared_error(y_lin, ridge_oof)))
ridge_cv_mae = float(mean_absolute_error(y_lin, ridge_oof))
svr_cv_rmse = float(np.sqrt(mean_squared_error(y_lin, svr_oof)))
svr_cv_mae = float(mean_absolute_error(y_lin, svr_oof))
fold_df = pd.DataFrame(fold_rows)

final_ridge_model = Ridge(alpha=10.0)
final_ridge_model.fit(X_lin_train_scaled, y_lin)
final_ridge_test_pred = np.clip(final_ridge_model.predict(X_lin_test_scaled), 0.0, 100.0)

final_svr_model = LinearSVR(C=0.1, epsilon=0.05, dual=False, loss='squared_epsilon_insensitive', random_state=42, max_iter=5000)
final_svr_model.fit(X_lin_train_scaled, y_lin)
final_svr_test_pred = np.clip(final_svr_model.predict(X_lin_test_scaled), 0.0, 100.0)

out_dir = Path('notebooks/reports/section5_ridge_linear_svr_legacy_features')
out_dir.mkdir(parents=True, exist_ok=True)
ridge_submission = pd.DataFrame({bundle['target_col']: final_ridge_test_pred})
svr_submission = pd.DataFrame({bundle['target_col']: final_svr_test_pred})
if 'id' in bundle['test'].columns:
    ridge_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: final_ridge_test_pred})
    svr_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: final_svr_test_pred})

ridge_submission.to_csv(out_dir / 'submission_ridge_legacy_features.csv', index=False)
svr_submission.to_csv(out_dir / 'submission_linear_svr_legacy_features.csv', index=False)
fold_df.to_csv(out_dir / 'ridge_linear_svr_legacy_features_folds.csv', index=False)
pd.DataFrame([
    {'model': 'ridge', 'cv_rmse': ridge_cv_rmse, 'cv_mae': ridge_cv_mae},
    {'model': 'linear_svr', 'cv_rmse': svr_cv_rmse, 'cv_mae': svr_cv_mae},
]).to_csv(out_dir / 'ridge_linear_svr_legacy_features_summary.csv', index=False)

print(f'Ridge CV RMSE     : {ridge_cv_rmse:.6f}')
print(f'Ridge CV MAE      : {ridge_cv_mae:.6f}')
print(f'LinearSVR CV RMSE : {svr_cv_rmse:.6f}')
print(f'LinearSVR CV MAE  : {svr_cv_mae:.6f}')
display(fold_df)
print('Enregistre :', out_dir.resolve())

Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
Taille du modele lineaire : X_train=(630000, 12), X_test=(270000, 12)
Ridge CV RMSE     : 9.194925
Ridge CV MAE      : 7.342287
LinearSVR CV RMSE : 9.194834
LinearSVR CV MAE  : 7.342109


,fold,ridge_rmse,ridge_mae,linear_svr_rmse,linear_svr_mae
0,1,9.178603,7.335218,9.178484,7.334994
1,2,9.197042,7.345773,9.196944,7.345620
2,3,9.209105,7.345871,9.209048,7.345713


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_ridge_linear_svr_legacy_features


In [ ]:
# 5.9 XGBoost + LinearSVR (Features Legacy)

XGBRegressor = require_xgboost_regressor()
bundle = build_legacy_feature_bundle(verbose=True)
X_mix_train = bundle['X_train'].copy()
X_mix_test = bundle['X_test'].copy()
y_mix = bundle['y'].copy().reset_index(drop=True)

print('Features utilisees :', bundle['feature_cols'])
print(f'Taille X_train : {X_mix_train.shape}, Taille X_test : {X_mix_test.shape}')

scaler = StandardScaler()
X_mix_train_scaled = scaler.fit_transform(X_mix_train)
X_mix_test_scaled = scaler.transform(X_mix_test)

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 3000,
    'learning_rate': 0.1,
    'max_depth': 5,
    'min_child_weight': 25,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'reg_alpha': 0.1,
    'reg_lambda': 5.0,
    'gamma': 1.0,
    'random_state': 42,
    'n_jobs': -1,
}

kf = KFold(n_splits=3, shuffle=True, random_state=42)
xgb_oof = np.zeros(len(X_mix_train), dtype=float)
svr_oof = np.zeros(len(X_mix_train), dtype=float)
fold_rows = []
xgb_best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_mix_train, y_mix), start=1):
    X_tr = X_mix_train.iloc[tr_idx]
    X_va = X_mix_train.iloc[va_idx]
    Xs_tr = X_mix_train_scaled[tr_idx]
    Xs_va = X_mix_train_scaled[va_idx]
    y_tr = y_mix.iloc[tr_idx]
    y_va = y_mix.iloc[va_idx]

    xgb_model = XGBRegressor(**xgb_params, early_stopping_rounds=50)
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_preds = np.clip(xgb_model.predict(X_va), 0.0, 100.0)
    xgb_oof[va_idx] = xgb_preds

    best_iter = getattr(xgb_model, 'best_iteration', None)
    if best_iter is not None:
        xgb_best_rounds.append(int(best_iter) + 1)

    svr_model = LinearSVR(C=10, epsilon=0.05, dual=False, loss='squared_epsilon_insensitive', random_state=42, max_iter=10000)
    svr_model.fit(Xs_tr, y_tr)
    svr_preds = np.clip(svr_model.predict(Xs_va), 0.0, 100.0)
    svr_oof[va_idx] = svr_preds

    fold_rows.append({
        'fold': fold_idx,
        'xgb_rmse': float(np.sqrt(mean_squared_error(y_va, xgb_preds))),
        'xgb_mae': float(mean_absolute_error(y_va, xgb_preds)),
        'linear_svr_rmse': float(np.sqrt(mean_squared_error(y_va, svr_preds))),
        'linear_svr_mae': float(mean_absolute_error(y_va, svr_preds)),
    })

xgb_cv_rmse = float(np.sqrt(mean_squared_error(y_mix, xgb_oof)))
xgb_cv_mae = float(mean_absolute_error(y_mix, xgb_oof))
svr_cv_rmse = float(np.sqrt(mean_squared_error(y_mix, svr_oof)))
svr_cv_mae = float(mean_absolute_error(y_mix, svr_oof))
fold_df = pd.DataFrame(fold_rows)

final_n_estimators = int(np.median(xgb_best_rounds)) if xgb_best_rounds else xgb_params['n_estimators']
final_xgb_params = xgb_params.copy()
final_xgb_params['n_estimators'] = final_n_estimators
final_xgb_model = XGBRegressor(**final_xgb_params)
final_xgb_model.fit(X_mix_train, y_mix)
final_xgb_test_pred = np.clip(final_xgb_model.predict(X_mix_test), 0.0, 100.0)

final_svr_model = LinearSVR(C=10, epsilon=0.05, dual=False, loss='squared_epsilon_insensitive', random_state=42, max_iter=10000)
final_svr_model.fit(X_mix_train_scaled, y_mix)
final_svr_test_pred = np.clip(final_svr_model.predict(X_mix_test_scaled), 0.0, 100.0)

out_dir = Path('notebooks/reports/section5_xgb_linear_svr_legacy_features')
out_dir.mkdir(parents=True, exist_ok=True)
xgb_submission = pd.DataFrame({bundle['target_col']: final_xgb_test_pred})
svr_submission = pd.DataFrame({bundle['target_col']: final_svr_test_pred})
if 'id' in bundle['test'].columns:
    xgb_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: final_xgb_test_pred})
    svr_submission = pd.DataFrame({'id': bundle['test']['id'], bundle['target_col']: final_svr_test_pred})

xgb_submission.to_csv(out_dir / 'submission_xgb_legacy_features.csv', index=False)
svr_submission.to_csv(out_dir / 'submission_linear_svr_legacy_features.csv', index=False)
fold_df.to_csv(out_dir / 'xgb_linear_svr_legacy_features_folds.csv', index=False)
pd.DataFrame([
    {'model': 'xgboost', 'cv_rmse': xgb_cv_rmse, 'cv_mae': xgb_cv_mae},
    {'model': 'linear_svr', 'cv_rmse': svr_cv_rmse, 'cv_mae': svr_cv_mae},
]).to_csv(out_dir / 'xgb_linear_svr_legacy_features_summary.csv', index=False)

print(f'XGBoost CV RMSE   : {xgb_cv_rmse:.6f}')
print(f'XGBoost CV MAE    : {xgb_cv_mae:.6f}')
print(f'LinearSVR CV RMSE : {svr_cv_rmse:.6f}')
print(f'LinearSVR CV MAE  : {svr_cv_mae:.6f}')
print(f'n_estimators final : {final_n_estimators}')
display(fold_df)
print('Enregistre :', out_dir.resolve())

Features utilisees : ['class_attendance', 'sleep_hours', 'sleep_quality', 'study_method', 'k3_f1', 'k3_f2', 'study_hours_x_sleep_hours', 'study_hours_x_sleep_quality', 'study_hours_x_facility_rating', 'study_hours_x_class_attendance', 'study_hours_x_class_attendance_x_sleep_hours', 'study_hours_sq']
Taille X_train : (630000, 12), Taille X_test : (270000, 12)
XGBoost CV RMSE   : 8.761582
XGBoost CV MAE    : 6.980743
LinearSVR CV RMSE : 9.194843
LinearSVR CV MAE  : 7.342124
n_estimators final : 1102


,fold,xgb_rmse,xgb_mae,linear_svr_rmse,linear_svr_mae
0,1,8.747761,6.974929,9.178492,7.335010
1,2,8.758390,6.981505,9.196957,7.345637
2,3,8.778567,6.985796,9.209053,7.345724


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_linear_svr_legacy_features


In [ ]:
# 5.10 XGBoost (Features SFS)


XGBRegressor = require_xgboost_regressor()
ctx = get_course_style_context(verbose=True)
X_sfs_train = np.asarray(ctx['X_scaled_sfs'], dtype=float)
X_sfs_test = np.asarray(ctx['X_test_scaled_sfs'], dtype=float)
y_sfs = ctx['y'].copy().reset_index(drop=True).astype(float)
selected_sfs_cols = ctx['selected_sfs_cols']

print('Features SFS utilisees :', list(selected_sfs_cols))
print(f'Tailles SFS : X_train={X_sfs_train.shape}, X_test={X_sfs_test.shape}')

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 15000,
    'learning_rate': 0.01,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.9,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.0,
    'reg_lambda': 4.0,
    'gamma': 0.0,
    'random_state': 42,
    'n_jobs': -1,
}

early_stopping_rounds = 100
kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_sfs_train), dtype=float)
fold_rows = []
best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_sfs_train, y_sfs), start=1):
    X_tr = X_sfs_train[tr_idx]
    X_va = X_sfs_train[va_idx]
    y_tr = y_sfs.iloc[tr_idx]
    y_va = y_sfs.iloc[va_idx]

    model = XGBRegressor(**xgb_params, early_stopping_rounds=early_stopping_rounds)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        best_rounds.append(int(best_iter) + 1)

    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

xgb_cv_rmse = float(np.sqrt(mean_squared_error(y_sfs, oof)))
xgb_cv_mae = float(mean_absolute_error(y_sfs, oof))
fold_df = pd.DataFrame(fold_rows)

final_n_estimators = int(np.median(best_rounds)) if best_rounds else xgb_params['n_estimators']
final_params = xgb_params.copy()
final_params['n_estimators'] = final_n_estimators
final_model = XGBRegressor(**final_params)
final_model.fit(X_sfs_train, y_sfs)
xgb_test_pred = np.clip(final_model.predict(X_sfs_test), 0.0, 100.0)

out_dir = Path('notebooks/reports/section5_xgb_sfs_features')
out_dir.mkdir(parents=True, exist_ok=True)
submission = pd.DataFrame({'exam_score': xgb_test_pred})
if 'id' in ctx['test'].columns:
    submission = pd.DataFrame({'id': ctx['test']['id'], 'exam_score': xgb_test_pred})
submission.to_csv(out_dir / 'submission_xgb_sfs_features.csv', index=False)
fold_df.to_csv(out_dir / 'xgb_sfs_features_folds.csv', index=False)
pd.DataFrame({'selected_sfs_cols': list(selected_sfs_cols)}).to_csv(out_dir / 'xgb_sfs_selected_columns.csv', index=False)

print(f'XGBoost SFS CV RMSE: {xgb_cv_rmse:.6f}')
print(f'XGBoost SFS CV MAE : {xgb_cv_mae:.6f}')
print(f'Final n_estimators : {final_n_estimators}')
display(fold_df)
print('Enregistre :', out_dir.resolve())

Features SFS utilisees : ['study_hours', 'class_attendance', 'sleep_quality', 'exam_difficulty', '_class_attendance_sin_12', '_class_attendance_sin_14', 'class_attendance_sq', 'study_sleep_ratio', 'magic_formula_2', 'study_hours_internet_access', 'TE_course', 'TE_study_method', 'TE_exam_difficulty', 'TE_study_hours_internet_access', 'TE_study_hours_study_method']
Tailles SFS : X_train=(630000, 15), X_test=(270000, 15)
XGBoost SFS CV RMSE: 8.695683
XGBoost SFS CV MAE : 6.918996
Final n_estimators : 14973


,fold,rmse,mae
0,1,8.687590,6.916454
1,2,8.692010,6.920448
2,3,8.707437,6.920085


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_sfs_features


In [ ]:
# 5.11 XGBoost Multi-Seed (Features SFS)


XGBRegressor = require_xgboost_regressor()
seeds = [42, 52, 62]
print('Seeds utilises :', seeds)

ctx = get_course_style_context(verbose=True)
X_sfs_train = np.asarray(ctx['X_scaled_sfs'], dtype=float)
X_sfs_test = np.asarray(ctx['X_test_scaled_sfs'], dtype=float)
y_sfs = ctx['y'].copy().reset_index(drop=True).astype(float)
selected_sfs_cols = ctx['selected_sfs_cols']

print('Features SFS utilisees :', list(selected_sfs_cols))
print(f'Tailles SFS : X_train={X_sfs_train.shape}, X_test={X_sfs_test.shape}')

base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 15000,
    'learning_rate': 0.01,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.9,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.0,
    'reg_lambda': 4.0,
    'gamma': 0.0,
    'n_jobs': -1,
}

early_stopping_rounds = 150
kf = KFold(n_splits=3, shuffle=True, random_state=42)
seed_summary_rows = []
seed_test_preds = []
seed_oofs = []
out_dir = Path('notebooks/reports/section5_xgb_sfs_multiseed')
out_dir.mkdir(parents=True, exist_ok=True)

for seed in seeds:
    print(f'Execution, seed={seed}')
    params = base_params.copy()
    params['random_state'] = seed

    oof = np.zeros(len(X_sfs_train), dtype=float)
    best_rounds = []
    fold_rows = []

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_sfs_train, y_sfs), start=1):
        X_tr = X_sfs_train[tr_idx]
        X_va = X_sfs_train[va_idx]
        y_tr = y_sfs.iloc[tr_idx]
        y_va = y_sfs.iloc[va_idx]

        model = XGBRegressor(**params, early_stopping_rounds=early_stopping_rounds)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        preds = np.clip(model.predict(X_va), 0.0, 100.0)
        oof[va_idx] = preds

        best_iter = getattr(model, 'best_iteration', None)
        if best_iter is not None:
            best_rounds.append(int(best_iter) + 1)

        fold_rows.append({
            'seed': seed,
            'fold': fold_idx,
            'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
            'mae': float(mean_absolute_error(y_va, preds)),
        })

    seed_rmse = float(np.sqrt(mean_squared_error(y_sfs, oof)))
    seed_mae = float(mean_absolute_error(y_sfs, oof))
    final_n_estimators = int(np.median(best_rounds)) if best_rounds else params['n_estimators']

    final_params = params.copy()
    final_params['n_estimators'] = final_n_estimators
    final_model = XGBRegressor(**final_params)
    final_model.fit(X_sfs_train, y_sfs)
    test_pred = np.clip(final_model.predict(X_sfs_test), 0.0, 100.0)

    seed_summary_rows.append({
        'seed': seed,
        'cv_rmse': seed_rmse,
        'cv_mae': seed_mae,
        'final_n_estimators': final_n_estimators,
    })
    seed_test_preds.append(test_pred)
    seed_oofs.append(oof)
    pd.DataFrame(fold_rows).to_csv(out_dir / f'xgb_sfs_multiseed_folds_seed_{seed}.csv', index=False)

seed_summary_df = pd.DataFrame(seed_summary_rows)
mean_test_pred = np.mean(np.column_stack(seed_test_preds), axis=1)
mean_oof = np.mean(np.column_stack(seed_oofs), axis=1)
ensemble_rmse = float(np.sqrt(mean_squared_error(y_sfs, mean_oof)))
ensemble_mae = float(mean_absolute_error(y_sfs, mean_oof))

submission = pd.DataFrame({'exam_score': mean_test_pred})
if 'id' in ctx['test'].columns:
    submission = pd.DataFrame({'id': ctx['test']['id'], 'exam_score': mean_test_pred})
submission.to_csv(out_dir / 'submission_xgb_sfs_multiseed.csv', index=False)
seed_summary_df.to_csv(out_dir / 'xgb_sfs_multiseed_summary.csv', index=False)
pd.DataFrame({'selected_sfs_cols': list(selected_sfs_cols)}).to_csv(out_dir / 'xgb_sfs_multiseed_selected_columns.csv', index=False)

print(seed_summary_df)
print(f'Multi-seed CV RMSE: {ensemble_rmse:.6f}')
print(f'Multi-seed CV MAE : {ensemble_mae:.6f}')
print('Enregistre :', out_dir.resolve())

Seeds utilises : [42, 52, 62]
Features SFS utilisees : ['study_hours', 'class_attendance', 'sleep_quality', 'exam_difficulty', '_class_attendance_sin_12', '_class_attendance_sin_14', 'class_attendance_sq', 'study_sleep_ratio', 'magic_formula_2', 'study_hours_internet_access', 'TE_course', 'TE_study_method', 'TE_exam_difficulty', 'TE_study_hours_internet_access', 'TE_study_hours_study_method']
Tailles SFS : X_train=(630000, 15), X_test=(270000, 15)
Execution, seed=42
Execution, seed=52
Execution, seed=62
   seed   cv_rmse    cv_mae  final_n_estimators
0    42  8.695434  6.918564               14998
1    52  8.695854  6.918666               14996
2    62  8.695819  6.918958               14991
Multi-seed CV RMSE: 8.695255
Multi-seed CV MAE : 6.918381
Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_sfs_multiseed


In [ ]:
# 5.12 XGBoost (12 Features SFS, Seed Unique)


XGBRegressor = require_xgboost_regressor()
ctx = get_course_style_context(verbose=True)
X_scaled = ctx['X_scaled']
X_test_scaled = ctx['X_test_scaled']
y_sfs12 = ctx['y'].copy().reset_index(drop=True).astype(float)

sfs12_feature_n = min(12, X_scaled.shape[1])
sfs12_sample_n = min(15000, len(X_scaled))

sfs12 = SequentialFeatureSelector(
    Ridge(alpha=1.0),
    n_features_to_select=sfs12_feature_n,
    direction='forward',
    cv=3,
    n_jobs=-1,
)
sfs12.fit(X_scaled.iloc[:sfs12_sample_n], y_sfs12.iloc[:sfs12_sample_n])

selected_sfs12_cols = X_scaled.columns[sfs12.get_support()].tolist()
X_sfs12_train = np.asarray(sfs12.transform(X_scaled), dtype=float)
X_sfs12_test = np.asarray(sfs12.transform(X_test_scaled), dtype=float)

print('Features SFS (12) utilisees :', selected_sfs12_cols)
print(f'Tailles SFS-12 : X_train={X_sfs12_train.shape}, X_test={X_sfs12_test.shape}')

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'n_estimators': 15000,
    'learning_rate': 0.01,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.9,
    'colsample_bytree': 1.0,
    'reg_alpha': 0.0,
    'reg_lambda': 4.0,
    'gamma': 0.0,
    'random_state': 42,
    'n_jobs': -1,
}

early_stopping_rounds = 100
kf = KFold(n_splits=3, shuffle=True, random_state=42)
oof = np.zeros(len(X_sfs12_train), dtype=float)
fold_rows = []
best_rounds = []

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_sfs12_train, y_sfs12), start=1):
    X_tr = X_sfs12_train[tr_idx]
    X_va = X_sfs12_train[va_idx]
    y_tr = y_sfs12.iloc[tr_idx]
    y_va = y_sfs12.iloc[va_idx]

    model = XGBRegressor(**xgb_params, early_stopping_rounds=early_stopping_rounds)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    preds = np.clip(model.predict(X_va), 0.0, 100.0)
    oof[va_idx] = preds

    best_iter = getattr(model, 'best_iteration', None)
    if best_iter is not None:
        best_rounds.append(int(best_iter) + 1)

    fold_rows.append({
        'fold': fold_idx,
        'rmse': float(np.sqrt(mean_squared_error(y_va, preds))),
        'mae': float(mean_absolute_error(y_va, preds)),
    })

xgb_cv_rmse = float(np.sqrt(mean_squared_error(y_sfs12, oof)))
xgb_cv_mae = float(mean_absolute_error(y_sfs12, oof))
fold_df = pd.DataFrame(fold_rows)

final_n_estimators = int(np.median(best_rounds)) if best_rounds else xgb_params['n_estimators']
final_params = xgb_params.copy()
final_params['n_estimators'] = final_n_estimators
final_model = XGBRegressor(**final_params)
final_model.fit(X_sfs12_train, y_sfs12)
xgb_test_pred = np.clip(final_model.predict(X_sfs12_test), 0.0, 100.0)

out_dir = Path('notebooks/reports/section5_xgb_sfs12_features')
out_dir.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({'exam_score': xgb_test_pred})
if 'id' in ctx['test'].columns:
    submission = pd.DataFrame({'id': ctx['test']['id'], 'exam_score': xgb_test_pred})

submission.to_csv(out_dir / 'submission_xgb_sfs12_features.csv', index=False)
fold_df.to_csv(out_dir / 'xgb_sfs12_features_folds.csv', index=False)
pd.DataFrame({'selected_sfs12_cols': selected_sfs12_cols}).to_csv(out_dir / 'xgb_sfs12_selected_columns.csv', index=False)

print(f'XGBoost SFS-12 CV RMSE: {xgb_cv_rmse:.6f}')
print(f'XGBoost SFS-12 CV MAE : {xgb_cv_mae:.6f}')
print(f'n_estimators final : {final_n_estimators}')
display(fold_df)
print('Enregistre :', out_dir.resolve())

Features SFS (12) utilisees : ['study_hours', 'sleep_quality', 'exam_difficulty', '_class_attendance_sin_12', '_class_attendance_sin_14', 'study_sleep_ratio', 'magic_formula_2', 'study_hours_internet_access', 'TE_course', 'TE_study_method', 'TE_study_hours_internet_access', 'TE_study_hours_study_method']
Tailles SFS-12 : X_train=(630000, 12), X_test=(270000, 12)
XGBoost SFS-12 CV RMSE: 8.715435
XGBoost SFS-12 CV MAE : 6.937442
n_estimators final : 14992


,fold,rmse,mae
0,1,8.708929,6.936567
1,2,8.711514,6.937886
2,3,8.725851,6.937872


Enregistre : /Users/selimkusursuz/Desktop/ML 2026/ML2026/notebooks/notebooks/reports/section5_xgb_sfs12_features
